# Topic Modeling of Patent-Cited and Policy-Cited Scholarly Literature

This notebook performs comparative topic modeling on two collections of
scholarly publications originating from the same broader Scopus research
corpus but identified through different forms of external citation:

1. **Patent-cited scholarly literature** — scholarly publications identified
   through Lens as being cited by patents.
2. **Policy-cited scholarly literature** — scholarly publications identified
   through Overton as being cited by policy documents.

The unit of analysis in both datasets is a **scholarly publication**. Patents
and policy documents themselves are not topic-modeled in this notebook.

The objective is to identify and compare the thematic structure of scholarly
research associated with patent uptake and policy uptake.

The analysis will include text preprocessing, document-term matrix
construction, LDA model selection, final topic estimation, topic
interpretation, prevalence analysis, cross-corpus thematic comparison, and
temporal analysis.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import subprocess
import tempfile


# ---------------------------------------------------------
# Project directories
# ---------------------------------------------------------

NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR


DATA_DIR = (
    PROJECT_DIR
    / "data"
)

LDA_DATA_DIR = (
    DATA_DIR
    / "lda"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "output"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "Project directory :",
    PROJECT_DIR
)

print(
    "Data directory    :",
    DATA_DIR
)

print(
    "LDA data directory:",
    LDA_DATA_DIR
)

print(
    "Output directory  :",
    OUTPUT_DIR
)

## 1. Load Patent-Cited and Policy-Cited Scholarly Data

Two scholarly publication datasets are loaded for independent topic modeling.

The **patent-cited corpus** contains publications from the broader Scopus
research corpus that were identified through Lens as being cited by patents.

The **policy-cited corpus** contains publications from the broader Scopus
research corpus that were identified through Overton as being cited by policy
documents.

Because both datasets contain scholarly publications, the comparison focuses
on differences in the thematic composition of research associated with two
forms of external uptake: technological uptake through patents and policy
uptake through policy-document citations.

The two collections are retained as separate corpora throughout the
topic-modeling workflow.

In [ ]:
# ---------------------------------------------------------
# Input files
# ---------------------------------------------------------

PATENT_CITED_FILE = (
    LDA_DATA_DIR
    / "lens_from_scopus_scholar.xlsx"
)

POLICY_CITED_FILE = (
    LDA_DATA_DIR
    / "overton_from_scopus_scholar.xlsx"
)


# ---------------------------------------------------------
# Validate files
# ---------------------------------------------------------

for input_file in [
    PATENT_CITED_FILE,
    POLICY_CITED_FILE,
]:

    if not input_file.exists():

        raise FileNotFoundError(
            f"Missing input file: "
            f"{input_file}"
        )


print(
    "Patent-cited file:",
    PATENT_CITED_FILE
)

print(
    "Policy-cited file:",
    POLICY_CITED_FILE
)

In [ ]:
# ---------------------------------------------------------
# Load datasets
# ---------------------------------------------------------

patent_cited = pd.read_excel(
    PATENT_CITED_FILE
)

policy_cited = pd.read_excel(
    POLICY_CITED_FILE
)


# ---------------------------------------------------------
# Basic dimensions
# ---------------------------------------------------------

print(
    "PATENT-CITED SCHOLARLY LITERATURE"
)

print(
    "--------------------------------"
)

print(
    f"Documents : "
    f"{len(patent_cited):,}"
)

print(
    f"Columns   : "
    f"{len(patent_cited.columns):,}"
)


print()


print(
    "POLICY-CITED SCHOLARLY LITERATURE"
)

print(
    "--------------------------------"
)

print(
    f"Documents : "
    f"{len(policy_cited):,}"
)

print(
    f"Columns   : "
    f"{len(policy_cited.columns):,}"
)

### 1.1 Validate Topic-Modeling Fields

Before constructing the topic-modeling corpora, the input datasets are checked
for the fields required by the subsequent analysis.

`Title` and `Abstract` are required for document interpretation and text
modeling. `DOI` is used as the principal publication identifier where
available, while `Year` is required for temporal analysis.

The validation step also reports missing abstracts and duplicate DOI values
without modifying the datasets. Any deduplication or exclusion decision is
performed explicitly in a subsequent step.

In [ ]:
# ---------------------------------------------------------
# Required fields
# ---------------------------------------------------------

REQUIRED_COLUMNS = [
    "Title",
    "Abstract",
    "DOI",
    "Year",
]


for corpus_name, corpus_df in {
    "patent_cited": patent_cited,
    "policy_cited": policy_cited,
}.items():

    missing_columns = [
        column
        for column in REQUIRED_COLUMNS
        if column not in corpus_df.columns
    ]

    if missing_columns:

        raise ValueError(
            f"{corpus_name} is missing "
            f"required columns: "
            f"{missing_columns}"
        )


print(
    "Required fields are available "
    "in both corpora."
)

In [ ]:
for corpus_name, corpus_df in {
    "patent_cited": patent_cited,
    "policy_cited": policy_cited,
}.items():

    display_name = (
        corpus_name
        .replace(
            "_",
            " "
        )
        .upper()
    )

    print()
    print(display_name)
    print("-" * len(display_name))

    print(
        f"Documents             : "
        f"{len(corpus_df):,}"
    )

    print(
        f"Non-missing titles    : "
        f"{corpus_df['Title'].notna().sum():,}"
    )

    print(
        f"Non-missing abstracts : "
        f"{corpus_df['Abstract'].notna().sum():,}"
    )

    print(
        f"Missing abstracts     : "
        f"{corpus_df['Abstract'].isna().sum():,}"
    )

    print(
        f"Non-missing DOI       : "
        f"{corpus_df['DOI'].notna().sum():,}"
    )

    print(
        f"Unique DOI            : "
        f"{corpus_df['DOI'].dropna().nunique():,}"
    )

    year_numeric = pd.to_numeric(
        corpus_df["Year"],
        errors="coerce",
    )

    print(
        f"Non-missing years     : "
        f"{year_numeric.notna().sum():,}"
    )

    if year_numeric.notna().any():

        print(
            f"Year range            : "
            f"{int(year_numeric.min())}"
            f"-"
            f"{int(year_numeric.max())}"
        )

### 1.2 Define the Independent Topic-Modeling Corpora

The patent-cited and policy-cited scholarly collections are treated as
independent topic-modeling corpora.

Each corpus will receive its own preprocessing, document-term matrix,
vocabulary filtering, LDA model-selection procedure, final topic model, and
topic interpretation.

The number of latent topics is not assumed to be identical across the two
corpora. Topic identifiers are also not directly comparable between
independently estimated LDA models. For example, Topic 1 in the patent-cited
model does not necessarily correspond to Topic 1 in the policy-cited model.

If direct thematic comparison is required after topic interpretation, a
higher-level common thematic mapping can subsequently be constructed.

In [ ]:
corpora = {
    "patent_cited":
        patent_cited.copy(),

    "policy_cited":
        policy_cited.copy(),
}


for corpus_name, corpus_df in (
    corpora.items()
):

    display_name = (
        corpus_name
        .replace(
            "_",
            " "
        )
        .upper()
    )

    print()
    print(display_name)
    print("-" * len(display_name))

    print(
        f"Documents          : "
        f"{len(corpus_df):,}"
    )

    print(
        f"Non-missing titles : "
        f"{corpus_df['Title'].notna().sum():,}"
    )

    print(
        f"Non-missing abstracts: "
        f"{corpus_df['Abstract'].notna().sum():,}"
    )

    print(
        f"Missing abstracts  : "
        f"{corpus_df['Abstract'].isna().sum():,}"
    )

### 1.3 Analysis Output Directories

Generated files are stored separately from the source data.

The patent-cited and policy-cited analyses share a dedicated output hierarchy
for preprocessing results, LDA models, topic interpretation, comparative
analysis, temporal analysis, and figures.

Intermediate outputs are saved so that expensive preprocessing and LDA
estimation steps do not need to be repeated unnecessarily.

In [ ]:
# ---------------------------------------------------------
# Patent-policy topic-modeling outputs
# ---------------------------------------------------------

ANALYSIS_OUTPUT_DIR = (
    OUTPUT_DIR
    / "patent_policy_topic_modeling"
)

R_PREPROCESS_DIR = (
    ANALYSIS_OUTPUT_DIR
    / "r_preprocessing"
)

LDA_DIR = (
    ANALYSIS_OUTPUT_DIR
    / "lda"
)

TOPIC_INTERPRETATION_DIR = (
    ANALYSIS_OUTPUT_DIR
    / "topic_interpretation"
)

TOPIC_PREVALENCE_DIR = (
    ANALYSIS_OUTPUT_DIR
    / "topic_prevalence"
)

COMPARISON_DIR = (
    ANALYSIS_OUTPUT_DIR
    / "cross_corpus_comparison"
)

TEMPORAL_DIR = (
    ANALYSIS_OUTPUT_DIR
    / "temporal_analysis"
)

FIGURE_DIR = (
    ANALYSIS_OUTPUT_DIR
    / "figures"
)


for directory in [
    ANALYSIS_OUTPUT_DIR,
    R_PREPROCESS_DIR,
    LDA_DIR,
    TOPIC_INTERPRETATION_DIR,
    TOPIC_PREVALENCE_DIR,
    COMPARISON_DIR,
    TEMPORAL_DIR,
    FIGURE_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


print(
    "Analysis output :",
    ANALYSIS_OUTPUT_DIR
)

print(
    "R preprocessing :",
    R_PREPROCESS_DIR
)

print(
    "LDA             :",
    LDA_DIR
)

print(
    "Interpretation  :",
    TOPIC_INTERPRETATION_DIR
)

print(
    "Prevalence      :",
    TOPIC_PREVALENCE_DIR
)

print(
    "Comparison      :",
    COMPARISON_DIR
)

print(
    "Temporal        :",
    TEMPORAL_DIR
)

print(
    "Figures         :",
    FIGURE_DIR
)

## 2. Prepare Corpora for Topic Modeling

The patent-cited and policy-cited datasets contain complete titles,
abstracts, DOIs, and publication years, with one unique DOI per scholarly
publication.

Because both collections were derived from the same broader Scopus research
corpus and subsequently selected according to external citation status, no
additional domain-relevance filter is applied at this stage. Applying a new
content filter after citation-based selection could alter the composition of
the patent-cited and policy-cited samples.

For topic modeling, the abstract is used as the primary textual unit. A
stable document identifier is assigned to each publication so that the
processed text, LDA document-topic probabilities, representative documents,
and original publication metadata can be linked consistently throughout the
analysis.

In [ ]:
# ---------------------------------------------------------
# Construct analysis corpora
# ---------------------------------------------------------

lda_corpora = {}

for corpus_name, corpus_df in (
    corpora.items()
):

    df = (
        corpus_df
        .copy()
        .reset_index(drop=True)
    )

    # Stable document identifier.
    # R uses 1-based indexing, so start at 1.
    df.insert(
        0,
        "document_id",
        np.arange(
            1,
            len(df) + 1,
        ),
    )

    # Ensure abstract is string and trimmed.
    df["Abstract"] = (
        df["Abstract"]
        .astype(str)
        .str.strip()
    )

    # Final safety check.
    valid_abstract = (
        df["Abstract"]
        .notna()
        & df["Abstract"].ne("")
    )

    if not valid_abstract.all():

        raise ValueError(
            f"{corpus_name} contains "
            f"{(~valid_abstract).sum()} "
            f"invalid abstracts."
        )

    lda_corpora[
        corpus_name
    ] = df


# ---------------------------------------------------------
# Diagnostics
# ---------------------------------------------------------

for corpus_name, corpus_df in (
    lda_corpora.items()
):

    display_name = (
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print()
    print(display_name)
    print("-" * len(display_name))

    print(
        "Documents :",
        f"{len(corpus_df):,}"
    )

    print(
        "Document IDs:",
        f"{corpus_df['document_id'].min()}"
        f"-"
        f"{corpus_df['document_id'].max()}"
    )

    print(
        "Unique DOI:",
        f"{corpus_df['DOI'].nunique():,}"
    )

### 2.1 Save Topic-Modeling Metadata

A metadata table is saved for each corpus before text preprocessing.

The metadata files preserve the relationship between the sequential
`document_id` used by the topic-modeling workflow and the original
publication information. These files are subsequently used to recover
titles, DOIs, publication years, and other metadata when identifying
representative publications for each latent topic.

In [ ]:
# ---------------------------------------------------------
# Metadata output directory
# ---------------------------------------------------------

METADATA_DIR = (
    ANALYSIS_OUTPUT_DIR
    / "metadata"
)

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------
# Save metadata
# ---------------------------------------------------------

for corpus_name, corpus_df in (
    lda_corpora.items()
):

    metadata_file = (
        METADATA_DIR
        / f"{corpus_name}_metadata.csv"
    )

    corpus_df.to_csv(
        metadata_file,
        index=False,
        encoding="utf-8",
    )

    print(
        f"{corpus_name.upper():15s}: "
        f"{metadata_file}"
    )

## 3. R Text-Processing Backend

Text preprocessing is performed in R to ensure that the document-term
matrices used for LDA are constructed using the same text-processing
implementation as the topic-modeling backend.

The same preprocessing procedure is applied independently to the
patent-cited and policy-cited corpora.

The preprocessing pipeline includes:

1. conversion to lowercase;
2. punctuation removal;
3. number removal;
4. whitespace normalization;
5. English stopword removal;
6. removal of query-specific terms; and
7. English stemming.

Applying an identical preprocessing procedure to both corpora ensures that
differences observed in the resulting topic models are not caused by
different text-processing rules.

In [ ]:
from pathlib import Path
import subprocess

RSCRIPT = Path(
    r"C:\Program Files\R\R-4.5.2\bin\Rscript.exe"
)

if not RSCRIPT.exists():
    raise FileNotFoundError(
        f"Rscript not found: {RSCRIPT}"
    )

# Check required R packages.
r_package_check = subprocess.run(
    [
        str(RSCRIPT),
        "-e",
        (
            'pkgs <- c("tm", "SnowballC", "slam", "topicmodels"); '
            'ok <- sapply(pkgs, requireNamespace, quietly=TRUE); '
            'cat(paste(pkgs, ok, sep="="), sep="\\n")'
        ),
    ],
    capture_output=True,
    text=True,
    check=True,
)

print("Rscript:")
print(RSCRIPT)

print("\nRequired R packages:")
print(r_package_check.stdout)

### 3.1 Text-Preprocessing Configuration

The same text-preprocessing procedure is applied to the patent-cited and
policy-cited scholarly corpora to maintain methodological consistency.

The preprocessing pipeline converts text to lowercase, removes punctuation
and numbers, removes standard English stopwords and query-specific terms,
applies English stemming, and normalizes whitespace.

Query-specific terms are removed because the two citation-selected corpora
originate from the same broader Scopus search. Terms imposed by the original
retrieval query are therefore expected to occur broadly by construction and
provide limited thematic discrimination.

In [ ]:
# ---------------------------------------------------------
# Shared preprocessing configuration
# ---------------------------------------------------------

MIN_TERM_LENGTH = 3

QUERY_STOPWORDS = [
    "power",
    "flow",
    "machine",
    "learning",
    "optimization",
    "optimisation",
]

print(
    "Minimum term length :",
    MIN_TERM_LENGTH
)

print(
    "Query stopwords     :",
    QUERY_STOPWORDS
)

### 3.2 Create Shared R Preprocessing Script

A single R preprocessing script is used for both corpora.

Using the same preprocessing implementation ensures that differences between
the patent-cited and policy-cited topic models are not introduced by
corpus-specific text-cleaning procedures.

The processed text retains the original `document_id`, allowing all later
DTM and LDA outputs to be linked back to the original scholarly publication.

In [ ]:
R_PREPROCESS_SCRIPT = (
    R_PREPROCESS_DIR
    / "preprocess_patent_policy_corpora.R"
)

r_preprocess_code = r'''
args <- commandArgs(
    trailingOnly = TRUE
)

input_file  <- args[1]
output_file <- args[2]

suppressPackageStartupMessages({
    library(tm)
    library(SnowballC)
})

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

query_stopwords <- c(
    "power",
    "flow",
    "machine",
    "learning",
    "optimization",
    "optimisation"
)

# ------------------------------------------------------------
# Load corpus
# ------------------------------------------------------------

data <- read.csv(
    input_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

if (!"document_id" %in% names(data)) {
    stop(
        "Input file does not contain document_id."
    )
}

if (!"Abstract" %in% names(data)) {
    stop(
        "Input file does not contain Abstract."
    )
}

abstracts <- data$Abstract

# ------------------------------------------------------------
# Construct corpus
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(
        abstracts
    )
)

# ------------------------------------------------------------
# Text preprocessing
# ------------------------------------------------------------

corpus <- tm_map(
    corpus,
    content_transformer(tolower)
)

corpus <- tm_map(
    corpus,
    removePunctuation
)

corpus <- tm_map(
    corpus,
    removeNumbers
)

# Explicitly remove ©
corpus <- tm_map(
    corpus,
    content_transformer(
        function(x) {
            gsub(
                "\u00A9",
                " ",
                x,
                fixed = TRUE
            )
        }
    )
)

corpus <- tm_map(
    corpus,
    stripWhitespace
)

corpus <- tm_map(
    corpus,
    removeWords,
    stopwords("english")
)

corpus <- tm_map(
    corpus,
    removeWords,
    query_stopwords
)

corpus <- tm_map(
    corpus,
    stemDocument,
    language = "english"
)

corpus <- tm_map(
    corpus,
    stripWhitespace
)

# ------------------------------------------------------------
# Extract processed text
# ------------------------------------------------------------

processed_text <- vapply(
    corpus,
    as.character,
    character(1)
)

output <- data.frame(
    document_id = data$document_id,
    processed_text = processed_text,
    stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

write.csv(
    output,
    output_file,
    row.names = FALSE,
    fileEncoding = "UTF-8"
)

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

cat(
    "Documents processed:",
    nrow(output),
    "\n"
)

cat(
    "Empty processed documents:",
    sum(
        !nzchar(
            trimws(
                output$processed_text
            )
        )
    ),
    "\n"
)
'''

R_PREPROCESS_SCRIPT.write_text(
    r_preprocess_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_PREPROCESS_SCRIPT
)

### 3.3 Preprocess Patent-Cited and Policy-Cited Abstracts

The abstract text from each corpus is passed independently through the shared
R preprocessing pipeline.

The resulting processed-text files are saved to disk so that later DTM and
LDA stages can be reproduced without repeating the text-preprocessing step.

In [ ]:
processed_corpora = {}


for corpus_name, corpus_df in (
    lda_corpora.items()
):

    # -----------------------------------------------------
    # Files
    # -----------------------------------------------------

    input_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_abstracts.csv"
    )

    output_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_processed.csv"
    )

    # -----------------------------------------------------
    # Save abstract input
    # -----------------------------------------------------

    corpus_df[
        [
            "document_id",
            "Abstract",
        ]
    ].to_csv(
        input_file,
        index=False,
        encoding="utf-8",
    )

    # -----------------------------------------------------
    # Run R
    # -----------------------------------------------------

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "-" * 40
    )

    preprocess_run = subprocess.run(
        [
            str(RSCRIPT),
            str(R_PREPROCESS_SCRIPT),
            str(input_file),
            str(output_file),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    print(
        "Return code:",
        preprocess_run.returncode
    )

    if preprocess_run.stdout.strip():

        print()
        print("R output:")

        print(
            preprocess_run.stdout
        )

    if preprocess_run.stderr.strip():

        print()
        print("R messages:")

        print(
            preprocess_run.stderr
        )

    if preprocess_run.returncode != 0:

        raise RuntimeError(
            f"R preprocessing failed "
            f"for {corpus_name}."
        )

    # -----------------------------------------------------
    # Reload processed corpus
    # -----------------------------------------------------

    processed_df = pd.read_csv(
        output_file
    )

    processed_corpora[
        corpus_name
    ] = processed_df

### 3.4 Validate Processed Corpora

The processed corpora are checked for document-count preservation and empty
documents.

The number and order of documents should remain unchanged during
preprocessing. Documents that become empty after stopword removal and
stemming would require special treatment before DTM construction.

In [ ]:
for corpus_name, processed_df in (
    processed_corpora.items()
):

    expected_documents = len(
        lda_corpora[
            corpus_name
        ]
    )

    empty_mask = (
        processed_df[
            "processed_text"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "-" * 40
    )

    print(
        "Expected documents :",
        f"{expected_documents:,}"
    )

    print(
        "Processed documents:",
        f"{len(processed_df):,}"
    )

    print(
        "Empty documents    :",
        f"{empty_mask.sum():,}"
    )

    print(
        "Document IDs match :",
        processed_df[
            "document_id"
        ].tolist()
        ==
        lda_corpora[
            corpus_name
        ][
            "document_id"
        ].tolist()
    )

### 3.5 Inspect Frequent Processed Terms

The most frequent processed terms are inspected before final DTM
construction.

This diagnostic helps identify residual retrieval terms, database
boilerplate, generic scholarly language, or other high-frequency terms that
may dominate the latent topics without providing useful thematic
information.

The purpose of this step is diagnostic rather than to automatically remove
frequent terms. Additional stopwords should only be introduced when their
removal is substantively justified and applied consistently to both corpora.

In [ ]:
from collections import Counter


TOP_N_TERMS = 30

frequent_term_tables = {}


for corpus_name, processed_df in (
    processed_corpora.items()
):

    tokens = []

    for text in (
        processed_df[
            "processed_text"
        ]
        .fillna("")
        .astype(str)
    ):

        tokens.extend(
            text.split()
        )

    counts = Counter(
        tokens
    )

    frequent_terms = pd.DataFrame(
        counts.most_common(
            TOP_N_TERMS
        ),
        columns=[
            "Term",
            "Frequency",
        ],
    )

    frequent_term_tables[
        corpus_name
    ] = frequent_terms

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 40
    )

    display(
        frequent_terms
    )

## 4. Document-Term Matrix Construction

Separate document-term matrices (DTMs) are constructed from the processed
patent-cited and policy-cited abstracts.

Each DTM represents scholarly publications as rows and processed vocabulary
terms as columns, with cell values corresponding to term frequencies.

Before constructing the final LDA input matrices, the raw vocabulary is
examined under alternative minimum document-frequency thresholds. This is
necessary because the patent-cited and policy-cited corpora differ in size
and are substantially smaller than the broader scholarly corpus from which
they were selected.

The document-frequency threshold is therefore selected from corpus-specific
diagnostics rather than inherited automatically from a previous analysis.

### 4.1 Raw Vocabulary Diagnostics

An initial DTM is constructed using all processed terms satisfying the
minimum term-length requirement.

For every vocabulary term, two quantities are recorded:

- **Term frequency (TF):** total number of occurrences across the corpus.
- **Document frequency (DF):** number of different documents containing
  the term.

The raw vocabulary diagnostics are subsequently used to evaluate alternative
rare-term filtering thresholds.

In [ ]:
R_DTM_DIAGNOSTIC_SCRIPT = (
    R_PREPROCESS_DIR
    / "dtm_diagnostics_patent_policy.R"
)

r_dtm_diagnostic_code = r'''
args <- commandArgs(
    trailingOnly = TRUE
)

processed_file <- args[1]
summary_file   <- args[2]
terms_file     <- args[3]

suppressPackageStartupMessages({
    library(tm)
    library(slam)
})

MIN_TERM_LENGTH <- 3

# ------------------------------------------------------------
# Load processed corpus
# ------------------------------------------------------------

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

valid <- (
    !is.na(processed)
    & nzchar(trimws(processed))
)

processed <- processed[valid]

# ------------------------------------------------------------
# Construct raw DTM
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(processed)
)

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

# ------------------------------------------------------------
# Frequencies
# ------------------------------------------------------------

document_frequency <- (
    slam::col_sums(
        dtm > 0
    )
)

term_frequency <- (
    slam::col_sums(
        dtm
    )
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

summary <- data.frame(
    Documents = dtm$nrow,
    Vocabulary = dtm$ncol,
    Tokens = sum(dtm$v),
    Min_Term_Length = MIN_TERM_LENGTH
)

# ------------------------------------------------------------
# Term diagnostics
# ------------------------------------------------------------

terms <- data.frame(
    Term = dtm$dimnames$Terms,
    Document_Frequency = as.numeric(
        document_frequency
    ),
    Term_Frequency = as.numeric(
        term_frequency
    ),
    stringsAsFactors = FALSE
)

terms <- terms[
    order(
        -terms$Document_Frequency,
        -terms$Term_Frequency
    ),
]

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

write.csv(
    summary,
    summary_file,
    row.names = FALSE
)

write.csv(
    terms,
    terms_file,
    row.names = FALSE
)

cat(
    "Documents:",
    dtm$nrow,
    "\n"
)

cat(
    "Raw vocabulary:",
    dtm$ncol,
    "\n"
)

cat(
    "Tokens:",
    sum(dtm$v),
    "\n"
)
'''

R_DTM_DIAGNOSTIC_SCRIPT.write_text(
    r_dtm_diagnostic_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_DTM_DIAGNOSTIC_SCRIPT
)

In [ ]:
raw_dtm_summaries = {}
raw_dtm_terms = {}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    processed_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_processed.csv"
    )

    summary_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_raw_dtm_summary.csv"
    )

    terms_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_raw_dtm_terms.csv"
    )

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "-" * 40
    )

    dtm_run = subprocess.run(
        [
            str(RSCRIPT),
            str(R_DTM_DIAGNOSTIC_SCRIPT),
            str(processed_file),
            str(summary_file),
            str(terms_file),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    print(
        "Return code:",
        dtm_run.returncode
    )

    if dtm_run.stdout.strip():

        print()
        print("R output:")

        print(
            dtm_run.stdout
        )

    if dtm_run.stderr.strip():

        print()
        print("R messages:")

        print(
            dtm_run.stderr
        )

    if dtm_run.returncode != 0:

        raise RuntimeError(
            f"DTM diagnostics failed "
            f"for {corpus_name}."
        )

    raw_dtm_summaries[
        corpus_name
    ] = pd.read_csv(
        summary_file
    )

    raw_dtm_terms[
        corpus_name
    ] = pd.read_csv(
        terms_file
    )

### 4.2 Minimum Document-Frequency Diagnostics

Terms occurring in very few documents can increase DTM sparsity and
introduce corpus-specific noise into LDA.

Alternative minimum document-frequency thresholds are therefore evaluated
before constructing the final DTMs.

Because the two corpora differ in size, thresholds are expressed as
percentages of the corresponding corpus size. This preserves a comparable
relative filtering rule while allowing the absolute minimum DF to differ
between the patent-cited and policy-cited corpora.

No threshold is selected automatically. The retained vocabulary under each
candidate threshold is inspected before fixing the final configuration.

In [ ]:
import math


DF_PERCENTAGES = [
    0.005,   # 0.5%
    0.010,   # 1.0%
    0.015,   # 1.5%
    0.020,   # 2.0%
    0.025,   # 2.5%
    0.030,   # 3.0%
]


df_diagnostic_tables = {}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    terms = (
        raw_dtm_terms[
            corpus_name
        ]
        .copy()
    )

    n_documents = len(
        lda_corpora[
            corpus_name
        ]
    )

    rows = []

    for fraction in DF_PERCENTAGES:

        minimum_df = math.ceil(
            n_documents
            * fraction
        )

        retained = (
            terms[
                "Document_Frequency"
            ]
            >= minimum_df
        )

        rows.append({
            "DF_Percent":
                fraction * 100,

            "Minimum_DF":
                minimum_df,

            "Retained_Terms":
                int(
                    retained.sum()
                ),

            "Removed_Terms":
                int(
                    (~retained).sum()
                ),

            "Vocabulary_Retained_Percent":
                (
                    100
                    * retained.sum()
                    / len(terms)
                ),
        })

    diagnostics = pd.DataFrame(
        rows
    )

    df_diagnostic_tables[
        corpus_name
    ] = diagnostics

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 40
    )

    display(
        diagnostics
    )

In [ ]:
for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    display(
        raw_dtm_summaries[
            corpus_name
        ]
    )

### 4.3 Final Document-Frequency Threshold

The document-frequency threshold was selected by examining the trade-off
between removal of low-frequency terms and preservation of sufficient lexical
diversity for topic discovery. A minimum document frequency of 1% of the
corresponding corpus size was adopted for both corpora.

At the 0.5% threshold, terms occurring in as few as four patent-cited
documents or two policy-cited documents were retained, resulting in
vocabularies of 1,899 and 2,147 terms, respectively. Such low-frequency terms
are more susceptible to representing document-specific terminology rather
than recurring corpus-level themes. Increasing the threshold to 1% raises the
minimum document frequency to eight patent-cited documents and four
policy-cited documents while retaining 1,295 and 1,371 terms, respectively.
Higher thresholds progressively reduce the available vocabulary; at 1.5%,
for example, the retained vocabularies decrease to 999 and 1,040 terms.

The 1% threshold was therefore used as a common proportional filtering
criterion that removes highly infrequent terms while preserving a sufficiently
rich vocabulary for identifying specialized themes in both corpora. This
threshold is a preprocessing choice rather than an intrinsic parameter of
LDA and is specific to the size and composition of the corpora analyzed here.

For corpus \(c\), the threshold is

$$
DF_{\min,c}
=
\left\lceil 0.01 N_c \right\rceil,
$$

where \(N_c\) denotes the number of documents in corpus \(c\). This gives

$$
DF_{\min,\mathrm{patent}}
=
\left\lceil 0.01 \times 778 \right\rceil
=
8,
$$

and

$$
DF_{\min,\mathrm{policy}}
=
\left\lceil 0.01 \times 392 \right\rceil
=
4.
$$

In [ ]:
FINAL_MIN_DOC_PERCENT = 0.01

final_min_doc_freq = {}

for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    n_documents = len(
        lda_corpora[
            corpus_name
        ]
    )

    min_df = math.ceil(
        n_documents
        * FINAL_MIN_DOC_PERCENT
    )

    final_min_doc_freq[
        corpus_name
    ] = min_df

    print(
        f"{corpus_name.upper():15s} | "
        f"Documents: {n_documents:,} | "
        f"Minimum DF: {min_df}"
    )

### 4.4 Construct Final Document-Term Matrices

The final document-term matrix for each corpus is reconstructed from the
processed abstracts using the selected corpus-specific minimum
document-frequency threshold.

Terms not meeting the threshold are removed before LDA estimation.

Documents that become empty after vocabulary filtering are identified
explicitly. The final DTM summary records the number of documents,
vocabulary size, token count, minimum document frequency, and number of
empty documents.

In [ ]:
R_FINAL_DTM_SCRIPT = (
    R_PREPROCESS_DIR
    / "construct_final_dtm_patent_policy.R"
)

r_final_dtm_code = r'''
args <- commandArgs(
    trailingOnly = TRUE
)

processed_file <- args[1]
summary_file   <- args[2]
terms_file     <- args[3]
min_doc_freq   <- as.integer(args[4])

suppressPackageStartupMessages({
    library(tm)
    library(slam)
})

MIN_TERM_LENGTH <- 3
MIN_DOC_FREQ <- min_doc_freq

# ------------------------------------------------------------
# Load processed documents
# ------------------------------------------------------------

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

valid <- (
    !is.na(processed)
    & nzchar(trimws(processed))
)

processed <- processed[valid]

# ------------------------------------------------------------
# Construct raw DTM
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(processed)
)

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

# ------------------------------------------------------------
# Apply document-frequency filtering
# ------------------------------------------------------------

document_frequency <- slam::col_sums(
    dtm > 0
)

keep_terms <- (
    document_frequency
    >= MIN_DOC_FREQ
)

dtm <- dtm[
    ,
    keep_terms
]

# ------------------------------------------------------------
# Check empty documents
# ------------------------------------------------------------

document_tokens <- slam::row_sums(
    dtm
)

empty_documents <- (
    document_tokens == 0
)

n_empty <- sum(
    empty_documents
)

# ------------------------------------------------------------
# Final frequencies
# ------------------------------------------------------------

final_document_frequency <- (
    slam::col_sums(
        dtm > 0
    )
)

final_term_frequency <- (
    slam::col_sums(
        dtm
    )
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

summary <- data.frame(
    Documents = dtm$nrow,
    Vocabulary = dtm$ncol,
    Tokens = sum(dtm$v),
    Minimum_DF = MIN_DOC_FREQ,
    Min_Term_Length = MIN_TERM_LENGTH,
    Empty_Documents = n_empty
)

terms <- data.frame(
    Term = dtm$dimnames$Terms,
    Document_Frequency = as.numeric(
        final_document_frequency
    ),
    Term_Frequency = as.numeric(
        final_term_frequency
    ),
    stringsAsFactors = FALSE
)

terms <- terms[
    order(
        -terms$Document_Frequency,
        -terms$Term_Frequency
    ),
]

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

write.csv(
    summary,
    summary_file,
    row.names = FALSE
)

write.csv(
    terms,
    terms_file,
    row.names = FALSE
)

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

cat(
    "Documents:",
    dtm$nrow,
    "\n"
)

cat(
    "Vocabulary:",
    dtm$ncol,
    "\n"
)

cat(
    "Tokens:",
    sum(dtm$v),
    "\n"
)

cat(
    "Minimum DF:",
    MIN_DOC_FREQ,
    "\n"
)

cat(
    "Empty documents:",
    n_empty,
    "\n"
)
'''

R_FINAL_DTM_SCRIPT.write_text(
    r_final_dtm_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_FINAL_DTM_SCRIPT
)

In [ ]:
final_dtm_summaries = {}
final_dtm_terms = {}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    processed_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_processed.csv"
    )

    summary_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_final_dtm_summary.csv"
    )

    terms_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_final_dtm_terms.csv"
    )

    min_df = (
        final_min_doc_freq[
            corpus_name
        ]
    )

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "-" * 40
    )

    print(
        "Minimum DF:",
        min_df
    )

    final_dtm_run = subprocess.run(
        [
            str(RSCRIPT),
            str(R_FINAL_DTM_SCRIPT),
            str(processed_file),
            str(summary_file),
            str(terms_file),
            str(min_df),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    print(
        "Return code:",
        final_dtm_run.returncode
    )

    if final_dtm_run.stdout.strip():

        print()
        print("R output:")

        print(
            final_dtm_run.stdout
        )

    if final_dtm_run.stderr.strip():

        print()
        print("R messages:")

        print(
            final_dtm_run.stderr
        )

    if final_dtm_run.returncode != 0:

        raise RuntimeError(
            f"Final DTM construction failed "
            f"for {corpus_name}."
        )

    final_dtm_summaries[
        corpus_name
    ] = pd.read_csv(
        summary_file
    )

    final_dtm_terms[
        corpus_name
    ] = pd.read_csv(
        terms_file
    )

### 4.5 Final DTM Summary

The final DTM dimensions and token counts are inspected before proceeding to
LDA model selection.

The principal checks are:

- all intended documents remain represented;
- no document becomes empty after vocabulary filtering;
- the retained vocabulary is sufficiently large for thematic discovery; and
- the relative vocabulary-filtering rule has been applied consistently to
  both corpora.

In [ ]:
for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 40
    )

    display(
        final_dtm_summaries[
            corpus_name
        ]
    )

## 5. LDA Topic Modeling and Model Selection

Separate Latent Dirichlet Allocation (LDA) models are estimated for the
patent-cited and policy-cited scholarly corpora.

Because the two corpora may contain different thematic structures, the
number of latent topics is selected independently for each corpus rather
than imposing a common topic number.

Model selection is performed in stages. First, each corpus is divided into
training and held-out subsets using a reproducible random split. A coarse
search over candidate topic numbers is then performed using relatively short
Gibbs-sampling chains and evaluated using held-out perplexity.

Candidate topic resolutions identified during the screening stage are
subsequently evaluated using longer chains and additional diagnostics,
including topic coherence, inter-topic similarity, and manual
interpretability.

The final number of topics is therefore selected using both quantitative
model diagnostics and substantive inspection of the resulting topics.

### 5.1 Reproducible Training and Held-Out Splits

Each corpus is divided into an 80% training subset and a 20% held-out subset.

The training documents are used to estimate candidate LDA models, while the
held-out documents are used to evaluate predictive performance through
perplexity.

A fixed random seed is used so that the same document split is retained
throughout all candidate-model evaluations.

In [ ]:
# ---------------------------------------------------------
# LDA model-selection configuration
# ---------------------------------------------------------

TRAIN_FRACTION = 0.80
RANDOM_SEED = 123

K_COARSE = [
    5,
    10,
    15,
    20,
    25,
    30,
    40,
]

print(
    "Training fraction :",
    TRAIN_FRACTION
)

print(
    "Random seed       :",
    RANDOM_SEED
)

print(
    "Coarse K values   :",
    K_COARSE
)

In [ ]:
# ---------------------------------------------------------
# Create reproducible train / held-out splits
# ---------------------------------------------------------

train_test_indices = {}

for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    # Independent reproducible RNG for each corpus.
    rng = np.random.default_rng(
        RANDOM_SEED
    )

    n_documents = len(
        lda_corpora[
            corpus_name
        ]
    )

    # -----------------------------------------------------
    # Random permutation
    # -----------------------------------------------------

    all_indices = np.arange(
        1,
        n_documents + 1,
    )

    shuffled = rng.permutation(
        all_indices
    )

    n_train = int(
        np.floor(
            TRAIN_FRACTION
            * n_documents
        )
    )

    train_id = np.sort(
        shuffled[
            :n_train
        ]
    )

    test_id = np.sort(
        shuffled[
            n_train:
        ]
    )

    train_test_indices[
        corpus_name
    ] = {
        "train": train_id,
        "test": test_id,
    }

    # -----------------------------------------------------
    # Save
    # -----------------------------------------------------

    train_file = (
        LDA_DIR
        / f"{corpus_name}_train_indices.csv"
    )

    test_file = (
        LDA_DIR
        / f"{corpus_name}_test_indices.csv"
    )

    pd.DataFrame({
        "train_id": train_id
    }).to_csv(
        train_file,
        index=False,
    )

    pd.DataFrame({
        "test_id": test_id
    }).to_csv(
        test_file,
        index=False,
    )

    # -----------------------------------------------------
    # Diagnostics
    # -----------------------------------------------------

    print()

    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "-" * 40
    )

    print(
        "Total documents   :",
        f"{n_documents:,}"
    )

    print(
        "Training documents:",
        f"{len(train_id):,}"
    )

    print(
        "Held-out documents:",
        f"{len(test_id):,}"
    )

    print(
        "Overlap           :",
        len(
            set(train_id)
            & set(test_id)
        )
    )

    print(
        "Complete coverage :",
        len(
            set(train_id)
            | set(test_id)
        )
        == n_documents
    )

### 5.2 Coarse Topic-Number Search

A coarse topic-number search is performed independently for the patent-cited
and policy-cited corpora.

For each candidate number of topics \(K\), an LDA model is estimated using
the training documents and evaluated on the held-out documents using
perplexity.

Lower held-out perplexity indicates better predictive performance. However,
perplexity is used only as an initial screening criterion and does not by
itself determine the final number of topics.

Relatively short Gibbs-sampling chains are used during this stage to keep
the screening computationally efficient. Candidate topic resolutions
identified here will subsequently be evaluated using longer chains,
additional topic-quality diagnostics, and manual interpretability.

In [ ]:
# ---------------------------------------------------------
# Coarse LDA search configuration
# ---------------------------------------------------------

COARSE_BURN_IN = 50
COARSE_ITERATIONS = 100
COARSE_THIN = 10

print(
    "K values   :",
    K_COARSE
)

print(
    "Burn-in    :",
    COARSE_BURN_IN
)

print(
    "Iterations :",
    COARSE_ITERATIONS
)

print(
    "Thin       :",
    COARSE_THIN
)

#### 5.2.1 Coarse-Search R Backend

The coarse-search backend reconstructs the filtered DTM for each corpus,
applies the previously saved training/held-out split, estimates an LDA model
for each candidate value of \(K\), and calculates held-out perplexity.

The vocabulary is determined from the training data before model estimation,
and documents that become empty after training-vocabulary alignment are
reported explicitly.

Results are written to disk after every completed candidate model so that
completed runs are preserved if execution is interrupted.

In [ ]:
R_COARSE_SEARCH_SCRIPT = (
    LDA_DIR
    / "coarse_k_search_patent_policy.R"
)

r_coarse_search_code = r'''
args <- commandArgs(
    trailingOnly = TRUE
)

processed_file <- args[1]
train_file     <- args[2]
test_file      <- args[3]
output_file    <- args[4]
min_doc_freq   <- as.integer(args[5])

suppressPackageStartupMessages({
    library(tm)
    library(slam)
    library(topicmodels)
})

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MIN_TERM_LENGTH <- 3

K_VALUES <- c(
    5,
    10,
    15,
    20,
    25,
    30,
    40
)

BURN_IN <- 50
ITERATIONS <- 100
THIN <- 10
RANDOM_SEED <- 123

# ------------------------------------------------------------
# Load processed documents
# ------------------------------------------------------------

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

if (any(is.na(processed))) {
    stop(
        "Processed corpus contains missing text."
    )
}

# ------------------------------------------------------------
# Reconstruct raw DTM
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(
        processed
    )
)

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

# ------------------------------------------------------------
# Apply final corpus-level DF filter
# ------------------------------------------------------------

document_frequency <- slam::col_sums(
    dtm > 0
)

keep_terms <- (
    document_frequency
    >= min_doc_freq
)

dtm <- dtm[
    ,
    keep_terms
]

# ------------------------------------------------------------
# Load fixed train/test indices
# ------------------------------------------------------------

train_id <- read.csv(
    train_file
)$train_id

test_id <- read.csv(
    test_file
)$test_id

# Validate split.
if (
    length(
        intersect(
            train_id,
            test_id
        )
    ) > 0
) {
    stop(
        "Training and test indices overlap."
    )
}

if (
    length(
        union(
            train_id,
            test_id
        )
    ) != dtm$nrow
) {
    stop(
        "Training/test indices do not cover all documents."
    )
}

# ------------------------------------------------------------
# Split DTM
# ------------------------------------------------------------

dtm_train <- dtm[
    train_id,
]

dtm_test <- dtm[
    test_id,
]

# ------------------------------------------------------------
# Restrict to terms occurring in training data
# ------------------------------------------------------------

training_term_mask <- (
    slam::col_sums(
        dtm_train
    ) > 0
)

dtm_train <- dtm_train[
    ,
    training_term_mask
]

dtm_test <- dtm_test[
    ,
    training_term_mask
]

# ------------------------------------------------------------
# Remove empty documents if necessary
# ------------------------------------------------------------

train_nonempty <- (
    slam::row_sums(
        dtm_train
    ) > 0
)

test_nonempty <- (
    slam::row_sums(
        dtm_test
    ) > 0
)

empty_train <- sum(
    !train_nonempty
)

empty_test <- sum(
    !test_nonempty
)

dtm_train <- dtm_train[
    train_nonempty,
]

dtm_test <- dtm_test[
    test_nonempty,
]

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

cat(
    "Training documents:",
    dtm_train$nrow,
    "\n"
)

cat(
    "Held-out documents:",
    dtm_test$nrow,
    "\n"
)

cat(
    "Vocabulary:",
    dtm_train$ncol,
    "\n"
)

cat(
    "Minimum DF:",
    min_doc_freq,
    "\n"
)

cat(
    "Empty training removed:",
    empty_train,
    "\n"
)

cat(
    "Empty held-out removed:",
    empty_test,
    "\n\n"
)

# ------------------------------------------------------------
# Candidate K search
# ------------------------------------------------------------

results <- data.frame(
    K = integer(),
    Perplexity = numeric(),
    Elapsed_seconds = numeric()
)

for (k in K_VALUES) {

    cat(
        "Fitting K =",
        k,
        "... "
    )

    flush.console()

    start_time <- Sys.time()

    model <- topicmodels::LDA(
        dtm_train,
        k = k,
        method = "Gibbs",
        control = list(
            seed = RANDOM_SEED,
            burnin = BURN_IN,
            iter = ITERATIONS,
            thin = THIN
        )
    )

    heldout_perplexity <- (
        topicmodels::perplexity(
            model,
            newdata = dtm_test
        )
    )

    elapsed_seconds <- as.numeric(
        difftime(
            Sys.time(),
            start_time,
            units = "secs"
        )
    )

    results <- rbind(
        results,
        data.frame(
            K = k,
            Perplexity =
                heldout_perplexity,
            Elapsed_seconds =
                elapsed_seconds
        )
    )

    # Save incrementally.
    write.csv(
        results,
        output_file,
        row.names = FALSE
    )

    cat(
        sprintf(
            "perplexity = %.4f | %.2f s\n",
            heldout_perplexity,
            elapsed_seconds
        )
    )

    flush.console()
}
'''

R_COARSE_SEARCH_SCRIPT.write_text(
    r_coarse_search_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_COARSE_SEARCH_SCRIPT
)

In [ ]:
# ---------------------------------------------------------
# Check whether coarse K search has already been completed
# ---------------------------------------------------------

for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    output_file = (
        LDA_DIR
        / f"{corpus_name}_coarse_k_search.csv"
    )

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 50
    )

    if output_file.exists():

        existing_results = pd.read_csv(
            output_file
        )

        print(
            "Result file : FOUND"
        )

        print(
            "File        :",
            output_file
        )

        print(
            "K values    :",
            existing_results[
                "K"
            ].tolist()
        )

        print(
            "Rows        :",
            len(
                existing_results
            )
        )

        display(
            existing_results
        )

    else:

        print(
            "Result file : NOT FOUND"
        )

# If the previous cell shows result of perplexity, don't run the coarse K search again. It takes a long time to run.

In [ ]:
coarse_results = {}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    processed_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_processed.csv"
    )

    train_file = (
        LDA_DIR
        / f"{corpus_name}_train_indices.csv"
    )

    test_file = (
        LDA_DIR
        / f"{corpus_name}_test_indices.csv"
    )

    output_file = (
        LDA_DIR
        / f"{corpus_name}_coarse_k_search.csv"
    )

    min_df = (
        final_min_doc_freq[
            corpus_name
        ]
    )

    print()
    print(
        "Running coarse K search..."
    )

    print(
        "Corpus     :",
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "Documents  :",
        f"{len(lda_corpora[corpus_name]):,}"
    )

    print(
        "Minimum DF :",
        min_df
    )

    print(
        "K values   :",
        K_COARSE
    )

    coarse_run = subprocess.run(
        [
            str(RSCRIPT),
            str(R_COARSE_SEARCH_SCRIPT),
            str(processed_file),
            str(train_file),
            str(test_file),
            str(output_file),
            str(min_df),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    print(
        "Return code:",
        coarse_run.returncode
    )

    if coarse_run.stdout.strip():

        print()
        print("R output:")

        print(
            coarse_run.stdout
        )

    if coarse_run.stderr.strip():

        print()
        print("R messages:")

        print(
            coarse_run.stderr
        )

    if coarse_run.returncode != 0:

        raise RuntimeError(
            f"Coarse K search failed "
            f"for {corpus_name}."
        )

    coarse_results[
        corpus_name
    ] = pd.read_csv(
        output_file
    )

    print()
    display(
        coarse_results[
            corpus_name
        ]
    )

### 5.3 Extended Topic-Number Search

The coarse search shows a continued reduction in held-out perplexity through
the largest tested topic resolution, \(K=40\), for both corpora.

Because no clear predictive plateau is observed within the initial search
range, the screening is extended to moderately larger topic resolutions.

The extended search is intentionally limited because the patent-cited and
policy-cited corpora contain substantially fewer documents than the broader
scholarly corpus. Increasing \(K\) excessively may produce highly fragmented
or weakly supported topics even when held-out perplexity continues to
decrease.

The extended search therefore evaluates \(K=50\) and \(K=60\). These results
are used only to characterize the continued perplexity trend. Final
topic-number selection will additionally consider semantic coherence,
inter-topic similarity, and substantive interpretability.

In [ ]:
K_EXTENDED = [
    50,
    60,
]

print(
    "Extended K values:",
    K_EXTENDED
)

#### 5.3.1 Extended-Search R Backend

The extended search uses the same training and held-out document splits,
vocabulary-filtering rules, random seed, and Gibbs-sampling configuration
as the coarse search.

Only the additional candidate topic resolutions are estimated, avoiding
unnecessary refitting of the models already evaluated during the coarse
search.

Results are saved separately for each corpus and later combined with the
coarse-search results.

In [ ]:
R_EXTENDED_SEARCH_SCRIPT = (
    LDA_DIR
    / "extended_k_search_patent_policy.R"
)

r_extended_search_code = r'''
args <- commandArgs(
    trailingOnly = TRUE
)

processed_file <- args[1]
train_file     <- args[2]
test_file      <- args[3]
output_file    <- args[4]
min_doc_freq   <- as.integer(args[5])

suppressPackageStartupMessages({
    library(tm)
    library(slam)
    library(topicmodels)
})

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MIN_TERM_LENGTH <- 3

K_VALUES <- c(
    50,
    60
)

BURN_IN <- 50
ITERATIONS <- 100
THIN <- 10
RANDOM_SEED <- 123

# ------------------------------------------------------------
# Load processed documents
# ------------------------------------------------------------

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

if (any(is.na(processed))) {
    stop(
        "Processed corpus contains missing text."
    )
}

# ------------------------------------------------------------
# Reconstruct DTM
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(
        processed
    )
)

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

# ------------------------------------------------------------
# Apply final corpus-level DF filter
# ------------------------------------------------------------

document_frequency <- slam::col_sums(
    dtm > 0
)

keep_terms <- (
    document_frequency
    >= min_doc_freq
)

dtm <- dtm[
    ,
    keep_terms
]

# ------------------------------------------------------------
# Load fixed train/test indices
# ------------------------------------------------------------

train_id <- read.csv(
    train_file
)$train_id

test_id <- read.csv(
    test_file
)$test_id

# ------------------------------------------------------------
# Validate split
# ------------------------------------------------------------

if (
    length(
        intersect(
            train_id,
            test_id
        )
    ) > 0
) {
    stop(
        "Training and test indices overlap."
    )
}

if (
    length(
        union(
            train_id,
            test_id
        )
    ) != dtm$nrow
) {
    stop(
        "Training/test indices do not cover all documents."
    )
}

# ------------------------------------------------------------
# Split DTM
# ------------------------------------------------------------

dtm_train <- dtm[
    train_id,
]

dtm_test <- dtm[
    test_id,
]

# ------------------------------------------------------------
# Retain terms represented in training data
# ------------------------------------------------------------

training_term_mask <- (
    slam::col_sums(
        dtm_train
    ) > 0
)

dtm_train <- dtm_train[
    ,
    training_term_mask
]

dtm_test <- dtm_test[
    ,
    training_term_mask
]

# ------------------------------------------------------------
# Remove empty documents if necessary
# ------------------------------------------------------------

train_nonempty <- (
    slam::row_sums(
        dtm_train
    ) > 0
)

test_nonempty <- (
    slam::row_sums(
        dtm_test
    ) > 0
)

empty_train <- sum(
    !train_nonempty
)

empty_test <- sum(
    !test_nonempty
)

dtm_train <- dtm_train[
    train_nonempty,
]

dtm_test <- dtm_test[
    test_nonempty,
]

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

cat(
    "Training documents:",
    dtm_train$nrow,
    "\n"
)

cat(
    "Held-out documents:",
    dtm_test$nrow,
    "\n"
)

cat(
    "Vocabulary:",
    dtm_train$ncol,
    "\n"
)

cat(
    "Minimum DF:",
    min_doc_freq,
    "\n"
)

cat(
    "Empty training removed:",
    empty_train,
    "\n"
)

cat(
    "Empty held-out removed:",
    empty_test,
    "\n\n"
)

# ------------------------------------------------------------
# Extended K search
# ------------------------------------------------------------

results <- data.frame(
    K = integer(),
    Perplexity = numeric(),
    Elapsed_seconds = numeric()
)

for (k in K_VALUES) {

    cat(
        "Fitting K =",
        k,
        "... "
    )

    flush.console()

    start_time <- Sys.time()

    model <- topicmodels::LDA(
        dtm_train,
        k = k,
        method = "Gibbs",
        control = list(
            seed = RANDOM_SEED,
            burnin = BURN_IN,
            iter = ITERATIONS,
            thin = THIN
        )
    )

    heldout_perplexity <- (
        topicmodels::perplexity(
            model,
            newdata = dtm_test
        )
    )

    elapsed_seconds <- as.numeric(
        difftime(
            Sys.time(),
            start_time,
            units = "secs"
        )
    )

    results <- rbind(
        results,
        data.frame(
            K = k,
            Perplexity =
                heldout_perplexity,
            Elapsed_seconds =
                elapsed_seconds
        )
    )

    # Save after each completed K.
    write.csv(
        results,
        output_file,
        row.names = FALSE
    )

    cat(
        sprintf(
            "perplexity = %.4f | %.2f s\n",
            heldout_perplexity,
            elapsed_seconds
        )
    )

    flush.console()
}
'''

R_EXTENDED_SEARCH_SCRIPT.write_text(
    r_extended_search_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_EXTENDED_SEARCH_SCRIPT
)

#### 5.3.2 Run Extended Topic-Number Search

The additional topic resolutions are estimated independently for the
patent-cited and policy-cited corpora.

Only \(K=50\) and \(K=60\) are evaluated in this stage because the models
for \(K=5\) through \(K=40\) have already been estimated during the coarse
search.

In [ ]:
# ---------------------------------------------------------
# Check existing extended K-search results
# ---------------------------------------------------------

for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    output_file = (
        LDA_DIR
        / f"{corpus_name}_extended_k_search.csv"
    )


    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 50
    )


    if not output_file.exists():

        print(
            "Result file : NOT FOUND"
        )

        print(
            "Expected    :",
            output_file
        )

        continue


    existing_results = pd.read_csv(
        output_file
    )


    print(
        "Result file : FOUND"
    )

    print(
        "File        :",
        output_file
    )

    print(
        "Rows        :",
        len(existing_results)
    )

    print(
        "K values    :",
        existing_results[
            "K"
        ]
        .astype(int)
        .tolist()
    )


    existing_k = set(
        existing_results[
            "K"
        ]
        .astype(int)
    )


    required_k = set(
        K_EXTENDED
    )


    if required_k.issubset(
        existing_k
    ):

        print(
            "Status      : COMPLETE"
        )

    else:

        print(
            "Status      : INCOMPLETE"
        )

        print(
            "Missing K   :",
            sorted(
                required_k
                - existing_k
            )
        )


    print()

    display(
        existing_results
        .sort_values(
            "K"
        )
        .reset_index(
            drop=True
        )
    )

# If the previous cell shows result of perplexity, don't run the next cell again. It takes a long time to run.

In [ ]:
extended_results = {}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    processed_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_processed.csv"
    )

    train_file = (
        LDA_DIR
        / f"{corpus_name}_train_indices.csv"
    )

    test_file = (
        LDA_DIR
        / f"{corpus_name}_test_indices.csv"
    )

    output_file = (
        LDA_DIR
        / f"{corpus_name}_extended_k_search.csv"
    )

    min_df = (
        final_min_doc_freq[
            corpus_name
        ]
    )

    print()
    print(
        "Running extended K search..."
    )

    print(
        "Corpus     :",
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "Documents  :",
        f"{len(lda_corpora[corpus_name]):,}"
    )

    print(
        "Minimum DF :",
        min_df
    )

    print(
        "K values   :",
        K_EXTENDED
    )

    extended_run = subprocess.run(
        [
            str(RSCRIPT),
            str(R_EXTENDED_SEARCH_SCRIPT),
            str(processed_file),
            str(train_file),
            str(test_file),
            str(output_file),
            str(min_df),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    print(
        "Return code:",
        extended_run.returncode
    )

    if extended_run.stdout.strip():

        print()
        print("R output:")

        print(
            extended_run.stdout
        )

    if extended_run.stderr.strip():

        print()
        print("R messages:")

        print(
            extended_run.stderr
        )

    if extended_run.returncode != 0:

        raise RuntimeError(
            f"Extended K search failed "
            f"for {corpus_name}."
        )

    extended_results[
        corpus_name
    ] = pd.read_csv(
        output_file
    )

    print()

    display(
        extended_results[
            corpus_name
        ]
    )

#### 5.4.1 Candidate-Evaluation Backend

Each candidate LDA model is refitted using a longer Gibbs-sampling chain than
was used during topic-number screening.

For every candidate \(K\), the evaluation records held-out perplexity,
topic-level semantic coherence, and pairwise topic similarity.

Semantic coherence is calculated from the document co-occurrence frequencies
of the highest-probability words in each topic. Inter-topic similarity is
calculated from cosine similarity between topic-word probability
distributions.

The complete topic-level diagnostics are saved in addition to the
model-level summaries so that poorly performing individual topics can be
identified during interpretation.

In [ ]:
R_CANDIDATE_EVALUATION_SCRIPT = (
    LDA_DIR
    / "candidate_evaluation_patent_policy.R"
)

r_candidate_evaluation_code = r'''
args <- commandArgs(
    trailingOnly = TRUE
)

processed_file <- args[1]
train_file     <- args[2]
test_file      <- args[3]
summary_file   <- args[4]
topic_file     <- args[5]
top_terms_file <- args[6]
min_doc_freq   <- as.integer(args[7])

suppressPackageStartupMessages({
    library(tm)
    library(slam)
    library(topicmodels)
})

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MIN_TERM_LENGTH <- 3

K_VALUES <- c(
    20,
    30,
    40,
    50
)

BURN_IN <- 500
ITERATIONS <- 2000
THIN <- 50

RANDOM_SEED <- 123

TOP_N <- 10

# ------------------------------------------------------------
# Load processed corpus
# ------------------------------------------------------------

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

if (any(is.na(processed))) {
    stop(
        "Processed corpus contains missing text."
    )
}

# ------------------------------------------------------------
# Construct DTM
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(
        processed
    )
)

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

# ------------------------------------------------------------
# Apply final document-frequency filter
# ------------------------------------------------------------

document_frequency <- slam::col_sums(
    dtm > 0
)

dtm <- dtm[
    ,
    document_frequency >= min_doc_freq
]

# ------------------------------------------------------------
# Fixed train/test split
# ------------------------------------------------------------

train_id <- read.csv(
    train_file
)$train_id

test_id <- read.csv(
    test_file
)$test_id

dtm_train <- dtm[
    train_id,
]

dtm_test <- dtm[
    test_id,
]

# ------------------------------------------------------------
# Restrict vocabulary to training data
# ------------------------------------------------------------

training_term_mask <- (
    slam::col_sums(
        dtm_train
    ) > 0
)

dtm_train <- dtm_train[
    ,
    training_term_mask
]

dtm_test <- dtm_test[
    ,
    training_term_mask
]

# ------------------------------------------------------------
# Remove empty documents
# ------------------------------------------------------------

train_nonempty <- (
    slam::row_sums(
        dtm_train
    ) > 0
)

test_nonempty <- (
    slam::row_sums(
        dtm_test
    ) > 0
)

n_empty_train <- sum(
    !train_nonempty
)

n_empty_test <- sum(
    !test_nonempty
)

dtm_train <- dtm_train[
    train_nonempty,
]

dtm_test <- dtm_test[
    test_nonempty,
]

# ------------------------------------------------------------
# Binary training DTM
# ------------------------------------------------------------

binary_train <- dtm_train

binary_train$v[] <- 1

# ------------------------------------------------------------
# Term document frequencies
# ------------------------------------------------------------

term_document_frequency <- (
    slam::col_sums(
        binary_train
    )
)

# ------------------------------------------------------------
# Term co-occurrence matrix
#
# C[i,j] = number of training documents in which
# terms i and j occur together.
# ------------------------------------------------------------

term_cooccurrence <- (
    slam::crossprod_simple_triplet_matrix(
        binary_train
    )
)

# ------------------------------------------------------------
# Semantic coherence
#
# Mimno-style coherence:
#
# sum_{m=2}^{M}
# sum_{l=1}^{m-1}
# log(
#     (D(v_m, v_l) + 1)
#     / D(v_l)
# )
# ------------------------------------------------------------

calculate_coherence <- function(
    top_indices,
    cooccurrence,
    term_df
) {

    score <- 0

    if (length(top_indices) < 2) {
        return(score)
    }

    for (m in 2:length(top_indices)) {

        for (l in 1:(m - 1)) {

            word_m <- top_indices[m]
            word_l <- top_indices[l]

            cooccur <- (
                cooccurrence[
                    word_m,
                    word_l
                ]
            )

            denominator <- (
                term_df[
                    word_l
                ]
            )

            if (denominator > 0) {

                score <- score + log(
                    (
                        cooccur + 1
                    )
                    /
                    denominator
                )
            }
        }
    }

    return(score)
}

# ------------------------------------------------------------
# Cosine similarity
# ------------------------------------------------------------

cosine_similarity <- function(
    x,
    y
) {

    denominator <- (
        sqrt(
            sum(
                x^2
            )
        )
        *
        sqrt(
            sum(
                y^2
            )
        )
    )

    if (denominator == 0) {
        return(0)
    }

    return(
        sum(
            x * y
        )
        /
        denominator
    )
}

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

cat(
    "Training documents:",
    dtm_train$nrow,
    "\n"
)

cat(
    "Held-out documents:",
    dtm_test$nrow,
    "\n"
)

cat(
    "Vocabulary:",
    dtm_train$ncol,
    "\n"
)

cat(
    "Minimum DF:",
    min_doc_freq,
    "\n"
)

cat(
    "Empty training removed:",
    n_empty_train,
    "\n"
)

cat(
    "Empty held-out removed:",
    n_empty_test,
    "\n\n"
)

# ------------------------------------------------------------
# Output containers
# ------------------------------------------------------------

summary_results <- data.frame()

topic_results <- data.frame()

top_term_results <- data.frame()

# ------------------------------------------------------------
# Candidate models
# ------------------------------------------------------------

for (k in K_VALUES) {

    cat(
        "Fitting K =",
        k,
        "... "
    )

    flush.console()

    start_time <- Sys.time()

    # --------------------------------------------------------
    # Fit LDA
    # --------------------------------------------------------

    model <- topicmodels::LDA(
        dtm_train,
        k = k,
        method = "Gibbs",
        control = list(
            seed = RANDOM_SEED,
            burnin = BURN_IN,
            iter = ITERATIONS,
            thin = THIN
        )
    )

    # --------------------------------------------------------
    # Held-out perplexity
    # --------------------------------------------------------

    heldout_perplexity <- (
        topicmodels::perplexity(
            model,
            newdata = dtm_test
        )
    )

    # --------------------------------------------------------
    # Topic-term probabilities
    # --------------------------------------------------------

    beta <- posterior(
        model
    )$terms

    # --------------------------------------------------------
    # Topic coherence
    # --------------------------------------------------------

    coherence_values <- numeric(
        k
    )

    for (topic in seq_len(k)) {

        top_indices <- order(
            beta[
                topic,
            ],
            decreasing = TRUE
        )[
            seq_len(
                min(
                    TOP_N,
                    ncol(beta)
                )
            )
        ]

        coherence_values[
            topic
        ] <- calculate_coherence(
            top_indices,
            term_cooccurrence,
            term_document_frequency
        )

        # ----------------------------------------------------
        # Save top terms
        # ----------------------------------------------------

        top_words <- colnames(
            beta
        )[
            top_indices
        ]

        for (
            rank in seq_along(
                top_words
            )
        ) {

            top_term_results <- rbind(
                top_term_results,
                data.frame(
                    K = k,
                    Topic = topic,
                    Rank = rank,
                    Term = top_words[
                        rank
                    ],
                    Beta = beta[
                        topic,
                        top_indices[
                            rank
                        ]
                    ]
                )
            )
        }
    }

    # --------------------------------------------------------
    # Pairwise topic similarity
    # --------------------------------------------------------

    similarity_values <- c()

    if (k > 1) {

        for (
            i in 1:(k - 1)
        ) {

            for (
                j in (i + 1):k
            ) {

                similarity_values <- c(
                    similarity_values,
                    cosine_similarity(
                        beta[
                            i,
                        ],
                        beta[
                            j,
                        ]
                    )
                )
            }
        }
    }

    # --------------------------------------------------------
    # Topic-level results
    # --------------------------------------------------------

    topic_results <- rbind(
        topic_results,
        data.frame(
            K = k,
            Topic = seq_len(k),
            Coherence =
                coherence_values
        )
    )

    # --------------------------------------------------------
    # Elapsed time
    # --------------------------------------------------------

    elapsed <- as.numeric(
        difftime(
            Sys.time(),
            start_time,
            units = "secs"
        )
    )

    # --------------------------------------------------------
    # Model-level summary
    # --------------------------------------------------------

    summary_results <- rbind(
        summary_results,
        data.frame(
            K = k,

            Perplexity =
                heldout_perplexity,

            Mean_Coherence =
                mean(
                    coherence_values
                ),

            Median_Coherence =
                median(
                    coherence_values
                ),

            Min_Coherence =
                min(
                    coherence_values
                ),

            Mean_Similarity =
                mean(
                    similarity_values
                ),

            Median_Similarity =
                median(
                    similarity_values
                ),

            Max_Similarity =
                max(
                    similarity_values
                ),

            Elapsed_seconds =
                elapsed
        )
    )

    # --------------------------------------------------------
    # Save incrementally
    # --------------------------------------------------------

    write.csv(
        summary_results,
        summary_file,
        row.names = FALSE
    )

    write.csv(
        topic_results,
        topic_file,
        row.names = FALSE
    )

    write.csv(
        top_term_results,
        top_terms_file,
        row.names = FALSE
    )

    # --------------------------------------------------------
    # Console output
    # --------------------------------------------------------

    cat(
        sprintf(
            paste0(
                "perplexity = %.4f",
                " | mean coherence = %.4f",
                " | mean similarity = %.4f",
                " | %.2f s\n"
            ),
            heldout_perplexity,
            mean(
                coherence_values
            ),
            mean(
                similarity_values
            ),
            elapsed
        )
    )

    flush.console()
}
'''

R_CANDIDATE_EVALUATION_SCRIPT.write_text(
    r_candidate_evaluation_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_CANDIDATE_EVALUATION_SCRIPT
)

#### 5.4.2 Run Candidate-Model Evaluation

The candidate topic resolutions are now estimated independently for the
patent-cited and policy-cited corpora using longer Gibbs-sampling chains.

For each candidate value of \(K\), the analysis records held-out perplexity,
semantic coherence, and inter-topic similarity. These diagnostics are used
to examine whether improvements in predictive fit are accompanied by
meaningful and sufficiently distinct latent topics.

The candidate models use the same fixed training and held-out document
splits established during the initial topic-number search.

In [ ]:
# ---------------------------------------------------------
# Candidate-model evaluation configuration
# ---------------------------------------------------------

K_CANDIDATES = [
    20,
    30,
    40,
    50,
]

CANDIDATE_BURN_IN = 500
CANDIDATE_ITERATIONS = 2000
CANDIDATE_THIN = 50
TOP_TERMS_COHERENCE = 10


print(
    "Candidate K values :",
    K_CANDIDATES
)

print(
    "Burn-in            :",
    CANDIDATE_BURN_IN
)

print(
    "Iterations         :",
    CANDIDATE_ITERATIONS
)

print(
    "Thin               :",
    CANDIDATE_THIN
)

print(
    "Top terms          :",
    TOP_TERMS_COHERENCE
)

In [ ]:
# ---------------------------------------------------------
# Check existing candidate-evaluation results
# ---------------------------------------------------------

for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    summary_file = (
        LDA_DIR
        / f"{corpus_name}_candidate_evaluation.csv"
    )

    topic_file = (
        LDA_DIR
        / f"{corpus_name}_candidate_topic_diagnostics.csv"
    )

    top_terms_file = (
        LDA_DIR
        / f"{corpus_name}_candidate_top_terms.csv"
    )


    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 60
    )


    # -----------------------------------------------------
    # File existence
    # -----------------------------------------------------

    files_exist = {
        "Summary":
            summary_file.exists(),

        "Topic diagnostics":
            topic_file.exists(),

        "Top terms":
            top_terms_file.exists(),
    }


    for name, exists in files_exist.items():

        print(
            f"{name:<20}:",
            "FOUND"
            if exists
            else "NOT FOUND"
        )


    # -----------------------------------------------------
    # Cannot be complete if any file is missing
    # -----------------------------------------------------

    if not all(
        files_exist.values()
    ):

        print(
            "Status              : INCOMPLETE"
        )

        continue


    # -----------------------------------------------------
    # Load files
    # -----------------------------------------------------

    summary = pd.read_csv(
        summary_file
    )

    topics = pd.read_csv(
        topic_file
    )

    top_terms = pd.read_csv(
        top_terms_file
    )


    # -----------------------------------------------------
    # Check K values
    # -----------------------------------------------------

    required_k = set(
        map(
            int,
            K_CANDIDATES,
        )
    )


    summary_k = set(
        summary[
            "K"
        ]
        .astype(int)
    )


    topics_k = set(
        topics[
            "K"
        ]
        .astype(int)
    )


    top_terms_k = set(
        top_terms[
            "K"
        ]
        .astype(int)
    )


    print()
    print(
        "Required K           :",
        sorted(
            required_k
        )
    )

    print(
        "Summary K            :",
        sorted(
            summary_k
        )
    )

    print(
        "Topic diagnostics K  :",
        sorted(
            topics_k
        )
    )

    print(
        "Top terms K          :",
        sorted(
            top_terms_k
        )
    )


    # -----------------------------------------------------
    # Missing K values
    # -----------------------------------------------------

    missing_summary = (
        required_k
        - summary_k
    )

    missing_topics = (
        required_k
        - topics_k
    )

    missing_top_terms = (
        required_k
        - top_terms_k
    )


    complete = (
        not missing_summary
        and
        not missing_topics
        and
        not missing_top_terms
    )


    print()

    if complete:

        print(
            "Status              : COMPLETE"
        )

    else:

        print(
            "Status              : INCOMPLETE"
        )

        print(
            "Missing summary K   :",
            sorted(
                missing_summary
            )
        )

        print(
            "Missing topic K     :",
            sorted(
                missing_topics
            )
        )

        print(
            "Missing top-term K  :",
            sorted(
                missing_top_terms
            )
        )


    # -----------------------------------------------------
    # Additional row-count checks
    # -----------------------------------------------------

    expected_topic_rows = sum(
        K_CANDIDATES
    )

    expected_top_term_rows = (
        sum(
            K_CANDIDATES
        )
        * TOP_TERMS_COHERENCE
    )


    print()
    print(
        "Summary rows         :",
        len(
            summary
        ),
        "/ expected",
        len(
            K_CANDIDATES
        ),
    )

    print(
        "Topic rows           :",
        len(
            topics
        ),
        "/ expected",
        expected_topic_rows,
    )

    print(
        "Top-term rows        :",
        len(
            top_terms
        ),
        "/ expected",
        expected_top_term_rows,
    )


    print()

    display(
        summary
        .sort_values(
            "K"
        )
        .reset_index(
            drop=True
        )
    )

# If the previous cell shows result of perplexity, don't run the next cell again. It takes a long time to run.

In [ ]:
candidate_results = {}
candidate_topic_diagnostics = {}
candidate_top_terms = {}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    # -----------------------------------------------------
    # Input files
    # -----------------------------------------------------

    processed_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_processed.csv"
    )

    train_file = (
        LDA_DIR
        / f"{corpus_name}_train_indices.csv"
    )

    test_file = (
        LDA_DIR
        / f"{corpus_name}_test_indices.csv"
    )

    # -----------------------------------------------------
    # Output files
    # -----------------------------------------------------

    summary_file = (
        LDA_DIR
        / f"{corpus_name}_candidate_evaluation.csv"
    )

    topic_file = (
        LDA_DIR
        / f"{corpus_name}_candidate_topic_diagnostics.csv"
    )

    top_terms_file = (
        LDA_DIR
        / f"{corpus_name}_candidate_top_terms.csv"
    )

    min_df = (
        final_min_doc_freq[
            corpus_name
        ]
    )

    # -----------------------------------------------------
    # Diagnostics
    # -----------------------------------------------------

    print()
    print(
        "Running candidate evaluation..."
    )

    print(
        "Corpus     :",
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "K values   :",
        K_CANDIDATES
    )

    print(
        "Minimum DF :",
        min_df
    )

    # -----------------------------------------------------
    # Run R
    # -----------------------------------------------------

    candidate_run = subprocess.run(
        [
            str(RSCRIPT),
            str(
                R_CANDIDATE_EVALUATION_SCRIPT
            ),
            str(processed_file),
            str(train_file),
            str(test_file),
            str(summary_file),
            str(topic_file),
            str(top_terms_file),
            str(min_df),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    print(
        "Return code:",
        candidate_run.returncode
    )

    if candidate_run.stdout.strip():

        print()
        print("R output:")

        print(
            candidate_run.stdout
        )

    if candidate_run.stderr.strip():

        print()
        print("R messages:")

        print(
            candidate_run.stderr
        )

    if candidate_run.returncode != 0:

        raise RuntimeError(
            f"Candidate evaluation failed "
            f"for {corpus_name}."
        )

    # -----------------------------------------------------
    # Load results
    # -----------------------------------------------------

    candidate_results[
        corpus_name
    ] = pd.read_csv(
        summary_file
    )

    candidate_topic_diagnostics[
        corpus_name
    ] = pd.read_csv(
        topic_file
    )

    candidate_top_terms[
        corpus_name
    ] = pd.read_csv(
        top_terms_file
    )

    # -----------------------------------------------------
    # Display model-level results
    # -----------------------------------------------------

    print()

    display(
        candidate_results[
            corpus_name
        ]
    )

## 5.5 Save Model-Selection Outputs

All model-selection outputs are saved to disk before proceeding to topic
interpretation.

This separates the computationally intensive LDA estimation stage from the
subsequent interpretive analysis. The saved outputs allow topic-number
selection, diagnostic visualization, and manual topic inspection to be
performed in a separate notebook without rerunning preprocessing or candidate
LDA estimation.

For each corpus, the saved outputs include:

- coarse topic-number search results;
- extended topic-number search results;
- candidate-model diagnostics;
- topic-level coherence diagnostics;
- top terms and topic-word probabilities for each candidate model; and
- fixed training and held-out document indices.

These files constitute the model-selection outputs used in the subsequent
topic-interpretation workflow.

# Skip this if you've saved the file

In [ ]:
# ---------------------------------------------------------
# Save / verify all model-selection outputs
# ---------------------------------------------------------

MODEL_SELECTION_DIR = (
    ANALYSIS_OUTPUT_DIR
    / "model_selection"
)

MODEL_SELECTION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    # -----------------------------------------------------
    # Candidate evaluation summary
    # -----------------------------------------------------

    candidate_summary_file = (
        MODEL_SELECTION_DIR
        / f"{corpus_name}_candidate_evaluation.csv"
    )

    candidate_results[
        corpus_name
    ].to_csv(
        candidate_summary_file,
        index=False,
    )

    # -----------------------------------------------------
    # Topic-level coherence
    # -----------------------------------------------------

    topic_diagnostics_file = (
        MODEL_SELECTION_DIR
        / f"{corpus_name}_candidate_topic_diagnostics.csv"
    )

    candidate_topic_diagnostics[
        corpus_name
    ].to_csv(
        topic_diagnostics_file,
        index=False,
    )

    # -----------------------------------------------------
    # Candidate top terms
    # -----------------------------------------------------

    top_terms_file = (
        MODEL_SELECTION_DIR
        / f"{corpus_name}_candidate_top_terms.csv"
    )

    candidate_top_terms[
        corpus_name
    ].to_csv(
        top_terms_file,
        index=False,
    )

    # -----------------------------------------------------
    # Coarse K search
    # -----------------------------------------------------

    coarse_file = (
        MODEL_SELECTION_DIR
        / f"{corpus_name}_coarse_k_search.csv"
    )

    coarse_results[
        corpus_name
    ].to_csv(
        coarse_file,
        index=False,
    )

    # -----------------------------------------------------
    # Extended K search
    # -----------------------------------------------------

    extended_file = (
        MODEL_SELECTION_DIR
        / f"{corpus_name}_extended_k_search.csv"
    )

    extended_results[
        corpus_name
    ].to_csv(
        extended_file,
        index=False,
    )

    # -----------------------------------------------------
    # Combined K search
    # -----------------------------------------------------

    combined = (
        pd.concat(
            [
                coarse_results[
                    corpus_name
                ],
                extended_results[
                    corpus_name
                ],
            ],
            ignore_index=True,
        )
        .drop_duplicates(
            subset="K",
            keep="last",
        )
        .sort_values("K")
        .reset_index(drop=True)
    )

    combined_file = (
        MODEL_SELECTION_DIR
        / f"{corpus_name}_combined_k_search.csv"
    )

    combined.to_csv(
        combined_file,
        index=False,
    )

    # -----------------------------------------------------
    # Train / test indices
    # -----------------------------------------------------

    train_indices_file = (
        MODEL_SELECTION_DIR
        / f"{corpus_name}_train_indices.csv"
    )

    test_indices_file = (
        MODEL_SELECTION_DIR
        / f"{corpus_name}_test_indices.csv"
    )

    pd.DataFrame({
        "train_id":
            train_test_indices[
                corpus_name
            ]["train"]
    }).to_csv(
        train_indices_file,
        index=False,
    )

    pd.DataFrame({
        "test_id":
            train_test_indices[
                corpus_name
            ]["test"]
    }).to_csv(
        test_indices_file,
        index=False,
    )

    # -----------------------------------------------------
    # Diagnostics
    # -----------------------------------------------------

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "-" * 50
    )

    print(
        "Candidate models :",
        len(
            candidate_results[
                corpus_name
            ]
        )
    )

    print(
        "Topic diagnostics:",
        len(
            candidate_topic_diagnostics[
                corpus_name
            ]
        )
    )

    print(
        "Top-term rows    :",
        len(
            candidate_top_terms[
                corpus_name
            ]
        )
    )

    print(
        "Saved to         :",
        MODEL_SELECTION_DIR
    )

In [ ]:
MODEL_SELECTION_EXCEL = (
    MODEL_SELECTION_DIR
    / "patent_policy_model_selection.xlsx"
)


with pd.ExcelWriter(
    MODEL_SELECTION_EXCEL,
    engine="openpyxl",
) as writer:

    for corpus_name in [
        "patent_cited",
        "policy_cited",
    ]:

        candidate_results[
            corpus_name
        ].to_excel(
            writer,
            sheet_name=(
                f"{corpus_name}_models"
            )[:31],
            index=False,
        )

        candidate_topic_diagnostics[
            corpus_name
        ].to_excel(
            writer,
            sheet_name=(
                f"{corpus_name}_topics"
            )[:31],
            index=False,
        )

        candidate_top_terms[
            corpus_name
        ].to_excel(
            writer,
            sheet_name=(
                f"{corpus_name}_terms"
            )[:31],
            index=False,
        )


print(
    "Saved:",
    MODEL_SELECTION_EXCEL
)

# Start here

In [ ]:
# ---------------------------------------------------------
# Reload existing candidate-evaluation results
# ---------------------------------------------------------

candidate_results = {}
candidate_topic_diagnostics = {}
candidate_top_terms = {}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    # -----------------------------------------------------
    # Existing result files
    # -----------------------------------------------------

    summary_file = (
        LDA_DIR
        / f"{corpus_name}_candidate_evaluation.csv"
    )

    topic_file = (
        LDA_DIR
        / f"{corpus_name}_candidate_topic_diagnostics.csv"
    )

    top_terms_file = (
        LDA_DIR
        / f"{corpus_name}_candidate_top_terms.csv"
    )


    # -----------------------------------------------------
    # Verify that all files exist
    # -----------------------------------------------------

    required_files = [
        summary_file,
        topic_file,
        top_terms_file,
    ]


    missing_files = [
        file
        for file in required_files
        if not file.exists()
    ]


    if missing_files:

        raise FileNotFoundError(
            "Missing candidate-evaluation files:\n"
            + "\n".join(
                str(file)
                for file
                in missing_files
            )
        )


    # -----------------------------------------------------
    # Load existing results
    # -----------------------------------------------------

    candidate_results[
        corpus_name
    ] = pd.read_csv(
        summary_file
    )


    candidate_topic_diagnostics[
        corpus_name
    ] = pd.read_csv(
        topic_file
    )


    candidate_top_terms[
        corpus_name
    ] = pd.read_csv(
        top_terms_file
    )


    # -----------------------------------------------------
    # Report
    # -----------------------------------------------------

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 60
    )

    print(
        "Candidate summary rows :",
        len(
            candidate_results[
                corpus_name
            ]
        )
    )

    print(
        "Topic diagnostic rows  :",
        len(
            candidate_topic_diagnostics[
                corpus_name
            ]
        )
    )

    print(
        "Top-term rows          :",
        len(
            candidate_top_terms[
                corpus_name
            ]
        )
    )

    print(
        "K values               :",
        sorted(
            candidate_results[
                corpus_name
            ][
                "K"
            ]
            .astype(int)
            .unique()
            .tolist()
        )
    )


print()
print(
    "Existing candidate-evaluation results loaded successfully."
)

In [ ]:
# ---------------------------------------------------------
# Candidate model comparison
# ---------------------------------------------------------

for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 60
    )

    display(
        candidate_results[
            corpus_name
        ]
        .sort_values(
            "K"
        )
        .reset_index(
            drop=True
        )
    )

In [ ]:
# ---------------------------------------------------------
# Candidate coherence distributions
# ---------------------------------------------------------

for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    coherence_summary = (
        candidate_topic_diagnostics[
            corpus_name
        ]
        .groupby(
            "K"
        )[
            "Coherence"
        ]
        .agg(
            [
                "count",
                "mean",
                "median",
                "std",
                "min",
                "max",
            ]
        )
        .reset_index()
    )


    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 60
    )

    display(
        coherence_summary
    )

In [ ]:
# ---------------------------------------------------------
# Build candidate topic-term summaries
# ---------------------------------------------------------

candidate_topic_summaries = {}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    terms = (
        candidate_top_terms[
            corpus_name
        ]
        .copy()
    )


    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 70
    )

    print(
        "Columns:",
        terms.columns.tolist()
    )

In [ ]:
# ---------------------------------------------------------
# Build readable candidate-topic summaries
# ---------------------------------------------------------

candidate_topic_summaries = {}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    terms = (
        candidate_top_terms[
            corpus_name
        ]
        .copy()
    )


    # -----------------------------------------------------
    # Combine the 10 top terms into one string per topic
    # -----------------------------------------------------

    topic_summary = (
        terms
        .sort_values(
            [
                "K",
                "Topic",
                "Rank",
            ]
        )
        .groupby(
            [
                "K",
                "Topic",
            ]
        )[
            "Term"
        ]
        .apply(
            lambda values:
                ", ".join(
                    values.astype(str)
                )
        )
        .reset_index(
            name="Top_Terms"
        )
    )


    # -----------------------------------------------------
    # Add topic coherence
    # -----------------------------------------------------

    topic_summary = (
        topic_summary
        .merge(
            candidate_topic_diagnostics[
                corpus_name
            ][
                [
                    "K",
                    "Topic",
                    "Coherence",
                ]
            ],
            on=[
                "K",
                "Topic",
            ],
            how="left",
        )
    )


    candidate_topic_summaries[
        corpus_name
    ] = (
        topic_summary
        .sort_values(
            [
                "K",
                "Topic",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 80
    )

    display(
        candidate_topic_summaries[
            corpus_name
        ]
    )

In [ ]:
# ---------------------------------------------------------
# Helper for inspecting candidate topics
# ---------------------------------------------------------

def show_candidate_topics(
    corpus_name,
    k,
):

    subset = (
        candidate_topic_summaries[
            corpus_name
        ]
        .loc[
            lambda df:
                df[
                    "K"
                ] == k
        ]
        .sort_values(
            "Topic"
        )
        .reset_index(
            drop=True
        )
    )


    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        f"K = {k}"
    )

    print(
        "=" * 80
    )


    display(
        subset
    )

In [ ]:
show_candidate_topics(
    "patent_cited",
    20,
)
show_candidate_topics(
    "patent_cited",
    30,
)

In [ ]:
show_candidate_topics(
    "policy_cited",
    20,
)
show_candidate_topics(
    "policy_cited",
    30,
)

In [ ]:
# ---------------------------------------------------------
# Compare complete K=20 and K=30 topic solutions
# ---------------------------------------------------------

for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    for k in [
        20,
        30,
    ]:

        subset = (
            candidate_topic_summaries[
                corpus_name
            ]
            .loc[
                lambda df:
                    df["K"] == k
            ]
            [
                [
                    "Topic",
                    "Top_Terms",
                    "Coherence",
                ]
            ]
            .sort_values(
                "Topic"
            )
            .reset_index(
                drop=True
            )
        )


        print()
        print(
            corpus_name
            .replace("_", " ")
            .upper()
        )

        print(
            f"K = {k}"
        )

        print(
            "=" * 100
        )


        print(
            subset.to_string(
                index=False
            )
        )

## 7. Final Patent- and Policy-Cited LDA Models

Following candidate-model evaluation, the final LDA models are estimated on
the complete patent-cited and policy-cited corpora. The train--test partitions
used for model selection are no longer applied at this stage because the
purpose of the final models is substantive topic interpretation rather than
held-out model evaluation.

A 20-topic solution is retained for each corpus. This provides a sufficiently
granular latent representation for subsequent cross-corpus comparison while
avoiding the greater fragmentation observed at higher topic counts. The
resulting latent topics are subsequently interpreted and, where appropriate,
consolidated into a smaller set of higher-level comparative themes.

In [ ]:
# ---------------------------------------------------------
# Final LDA configuration
# ---------------------------------------------------------

FINAL_K = {
    "patent_cited": 20,
    "policy_cited": 20,
}


FINAL_RANDOM_SEED = 123

FINAL_BURN_IN = 500

FINAL_ITERATIONS = 2000

FINAL_THIN = 50

FINAL_TOP_N = 10

FINAL_REPRESENTATIVE_N = 5


print("FINAL PATENT/POLICY LDA CONFIGURATION")
print("=" * 60)


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "-" * 40
    )

    print(
        "Documents       :",
        f"{len(lda_corpora[corpus_name]):,}"
    )

    print(
        "Final K         :",
        FINAL_K[
            corpus_name
        ]
    )

    print(
        "Minimum DF      :",
        final_min_doc_freq[
            corpus_name
        ]
    )


print()
print(
    "Random seed     :",
    FINAL_RANDOM_SEED
)

print(
    "Burn-in         :",
    FINAL_BURN_IN
)

print(
    "Iterations      :",
    FINAL_ITERATIONS
)

print(
    "Thin            :",
    FINAL_THIN
)

print(
    "Top terms       :",
    FINAL_TOP_N
)

print(
    "Representative  :",
    FINAL_REPRESENTATIVE_N
)

In [ ]:
# ---------------------------------------------------------
# Create final full-corpus LDA R script
# ---------------------------------------------------------

R_FINAL_LDA_SCRIPT = (
    R_PREPROCESS_DIR
    / "fit_final_lda_patent_policy.R"
)


r_final_lda_code = r'''
args <- commandArgs(trailingOnly = TRUE)


# ============================================================
# Arguments
# ============================================================

processed_file       <- args[1]
metadata_file        <- args[2]
summary_file         <- args[3]
top_terms_file       <- args[4]
theta_file           <- args[5]
document_topics_file <- args[6]
prevalence_file      <- args[7]
representative_file  <- args[8]

final_k         <- as.integer(args[9])
min_doc_freq    <- as.integer(args[10])
random_seed     <- as.integer(args[11])
burn_in         <- as.integer(args[12])
iterations      <- as.integer(args[13])
thin            <- as.integer(args[14])
top_n           <- as.integer(args[15])
representative_n <- as.integer(args[16])


# ============================================================
# Packages
# ============================================================

suppressPackageStartupMessages({
    library(tm)
    library(slam)
    library(topicmodels)
})


# ============================================================
# Configuration
# ============================================================

MIN_TERM_LENGTH <- 3

set.seed(random_seed)


# ============================================================
# Validate input files
# ============================================================

if (!file.exists(processed_file)) {
    stop(
        paste(
            "Processed corpus not found:",
            processed_file
        )
    )
}


if (!file.exists(metadata_file)) {
    stop(
        paste(
            "Metadata file not found:",
            metadata_file
        )
    )
}


# ============================================================
# Read processed corpus and metadata
# ============================================================

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)


metadata <- read.csv(
    metadata_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)


# ============================================================
# Validate required processed-text column
# ============================================================

if (!("processed_text" %in% colnames(data))) {
    stop(
        paste(
            "processed_text column not found in:",
            processed_file
        )
    )
}


# ============================================================
# Validate initial row alignment
# ============================================================

if (nrow(data) != nrow(metadata)) {
    stop(
        paste(
            "Processed corpus and metadata have different",
            "numbers of rows:",
            nrow(data),
            "versus",
            nrow(metadata)
        )
    )
}


cat(
    "Processed rows:",
    nrow(data),
    "\n"
)

cat(
    "Metadata rows:",
    nrow(metadata),
    "\n"
)


# ============================================================
# Identify valid processed documents
# ============================================================

processed <- data[["processed_text"]]


valid <- (
    !is.na(processed)
    &
    nzchar(trimws(processed))
)


n_invalid <- sum(!valid)


cat(
    "Empty processed documents before DTM:",
    n_invalid,
    "\n"
)


# ------------------------------------------------------------
# Preserve original row number BEFORE filtering
# ------------------------------------------------------------

original_row <- seq_len(nrow(metadata))


# ------------------------------------------------------------
# Apply the same validity mask everywhere
# ------------------------------------------------------------

data <- data[
    valid,
    ,
    drop = FALSE
]


metadata <- metadata[
    valid,
    ,
    drop = FALSE
]


processed <- processed[valid]


metadata$Original_Row <- original_row[valid]


cat(
    "Documents before final DTM:",
    length(processed),
    "\n"
)


# ============================================================
# Construct raw full-corpus DTM
# ============================================================

corpus <- VCorpus(
    VectorSource(processed)
)


dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)


cat(
    "Raw vocabulary:",
    ncol(dtm),
    "\n"
)


cat(
    "Raw tokens:",
    sum(dtm$v),
    "\n"
)


# ============================================================
# Apply minimum document-frequency filter
# ============================================================

document_frequency <- slam::col_sums(
    dtm > 0
)


keep_terms <- (
    document_frequency
    >=
    min_doc_freq
)


n_removed_terms <- sum(
    !keep_terms
)


dtm <- dtm[
    ,
    keep_terms
]


cat(
    "Minimum document frequency:",
    min_doc_freq,
    "\n"
)


cat(
    "Terms removed by DF filter:",
    n_removed_terms,
    "\n"
)


cat(
    "Vocabulary after DF filtering:",
    ncol(dtm),
    "\n"
)


# ============================================================
# Identify documents that become empty after DF filtering
# ============================================================

document_tokens <- slam::row_sums(
    dtm
)


nonempty <- (
    document_tokens > 0
)


n_empty <- sum(
    !nonempty
)


if (n_empty > 0) {

    dtm <- dtm[
        nonempty,
        ,
        drop = FALSE
    ]

    data <- data[
        nonempty,
        ,
        drop = FALSE
    ]

    metadata <- metadata[
        nonempty,
        ,
        drop = FALSE
    ]

}


cat(
    "Empty documents removed after DF filtering:",
    n_empty,
    "\n"
)


cat(
    "Final documents:",
    nrow(dtm),
    "\n"
)


cat(
    "Final vocabulary:",
    ncol(dtm),
    "\n"
)


cat(
    "Final tokens:",
    sum(dtm$v),
    "\n"
)


# ============================================================
# Final DTM validation
# ============================================================

if (nrow(dtm) == 0) {
    stop(
        "Final DTM contains zero documents."
    )
}


if (ncol(dtm) == 0) {
    stop(
        "Final DTM contains zero terms."
    )
}


if (nrow(dtm) != nrow(metadata)) {
    stop(
        paste(
            "DTM/metadata alignment failure:",
            nrow(dtm),
            "DTM rows versus",
            nrow(metadata),
            "metadata rows."
        )
    )
}


# ============================================================
# Fit final full-corpus LDA
# ============================================================

cat("\n")

cat(
    "Fitting final K =",
    final_k,
    "LDA model on full corpus...\n"
)


cat(
    "Seed:",
    random_seed,
    "\n"
)


cat(
    "Burn-in:",
    burn_in,
    "\n"
)


cat(
    "Iterations:",
    iterations,
    "\n"
)


cat(
    "Thin:",
    thin,
    "\n"
)


cat("\n")


start_time <- Sys.time()


final_model <- LDA(
    dtm,
    k = final_k,
    method = "Gibbs",
    control = list(
        seed = random_seed,
        burnin = burn_in,
        iter = iterations,
        thin = thin
    )
)


elapsed_seconds <- as.numeric(
    difftime(
        Sys.time(),
        start_time,
        units = "secs"
    )
)


# ============================================================
# Extract posterior distributions
# ============================================================

posterior_values <- posterior(
    final_model
)


theta <- posterior_values$topics

beta <- posterior_values$terms


# ============================================================
# Validate posterior dimensions
# ============================================================

if (nrow(theta) != nrow(dtm)) {
    stop(
        "Theta/document dimension mismatch."
    )
}


if (ncol(theta) != final_k) {
    stop(
        "Theta/topic dimension mismatch."
    )
}


if (nrow(beta) != final_k) {
    stop(
        "Beta/topic dimension mismatch."
    )
}


if (ncol(beta) != ncol(dtm)) {
    stop(
        "Beta/vocabulary dimension mismatch."
    )
}


# ============================================================
# Probability validation
# ============================================================

theta_error <- max(
    abs(
        rowSums(theta)
        - 1
    )
)


beta_error <- max(
    abs(
        rowSums(beta)
        - 1
    )
)


cat(
    "Theta max row-sum error:",
    theta_error,
    "\n"
)


cat(
    "Beta max row-sum error:",
    beta_error,
    "\n"
)


# ============================================================
# Topic prevalence
# ============================================================

topic_prevalence <- colMeans(
    theta
)


topic_prevalence <- (
    topic_prevalence
    /
    sum(topic_prevalence)
)


prevalence <- data.frame(
    Topic = seq_len(final_k),

    Prevalence = as.numeric(
        topic_prevalence
    ),

    Prevalence_Percent = (
        100
        *
        as.numeric(topic_prevalence)
    ),

    stringsAsFactors = FALSE
)


prevalence <- prevalence[
    order(
        -prevalence$Prevalence
    ),
    ,
    drop = FALSE
]


prevalence$Prevalence_Rank <- seq_len(
    nrow(prevalence)
)


prevalence <- prevalence[
    ,
    c(
        "Topic",
        "Prevalence_Rank",
        "Prevalence",
        "Prevalence_Percent"
    ),
    drop = FALSE
]


# ============================================================
# Extract top terms
# ============================================================

top_term_rows <- list()

row_index <- 1


for (topic_index in seq_len(final_k)) {

    term_order <- order(
        beta[
            topic_index,
        ],
        decreasing = TRUE
    )


    n_selected <- min(
        top_n,
        length(term_order)
    )


    selected <- term_order[
        seq_len(n_selected)
    ]


    for (rank_index in seq_along(selected)) {

        term_index <- selected[
            rank_index
        ]


        top_term_rows[[
            row_index
        ]] <- data.frame(
            Topic = topic_index,

            Rank = rank_index,

            Term = colnames(beta)[
                term_index
            ],

            Beta = beta[
                topic_index,
                term_index
            ],

            stringsAsFactors = FALSE
        )


        row_index <- row_index + 1

    }

}


top_terms <- do.call(
    rbind,
    top_term_rows
)


# ============================================================
# Complete document-topic probability matrix
# ============================================================

theta_output <- data.frame(
    Document_ID = seq_len(
        nrow(theta)
    ),

    Original_Row = metadata$Original_Row,

    theta,

    check.names = FALSE
)


colnames(theta_output)[
    -(1:2)
] <- paste0(
    "Topic_",
    seq_len(final_k)
)


# ============================================================
# Dominant topic for every document
# ============================================================

dominant_topic <- apply(
    theta,
    1,
    which.max
)


dominant_probability <- apply(
    theta,
    1,
    max
)


document_topics <- data.frame(
    Document_ID = seq_len(
        nrow(theta)
    ),

    Original_Row = metadata$Original_Row,

    Topic = dominant_topic,

    Topic_Probability = dominant_probability,

    stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# Attach metadata
# ------------------------------------------------------------

metadata_columns <- setdiff(
    colnames(metadata),
    "Original_Row"
)


document_topics <- cbind(
    document_topics,
    metadata[
        ,
        metadata_columns,
        drop = FALSE
    ]
)


# ============================================================
# Representative documents
# ============================================================

representative_rows <- list()

row_index <- 1


for (topic_index in seq_len(final_k)) {

    topic_probabilities <- theta[
        ,
        topic_index
    ]


    document_order <- order(
        topic_probabilities,
        decreasing = TRUE
    )


    n_selected <- min(
        representative_n,
        length(document_order)
    )


    selected_documents <- document_order[
        seq_len(n_selected)
    ]


    for (
        rank_index
        in seq_along(selected_documents)
    ) {

        document_index <- selected_documents[
            rank_index
        ]


        representative_row <- data.frame(
            Topic = topic_index,

            Rank = rank_index,

            Document_ID = document_index,

            Original_Row = metadata$Original_Row[
                document_index
            ],

            Topic_Probability = theta[
                document_index,
                topic_index
            ],

            stringsAsFactors = FALSE
        )


        representative_row <- cbind(
            representative_row,

            metadata[
                document_index,
                metadata_columns,
                drop = FALSE
            ]
        )


        representative_rows[[
            row_index
        ]] <- representative_row


        row_index <- row_index + 1

    }

}


representative_documents <- do.call(
    rbind,
    representative_rows
)


# ============================================================
# Model summary
# ============================================================

model_summary <- data.frame(
    K = final_k,

    Documents = nrow(dtm),

    Vocabulary = ncol(dtm),

    Tokens = sum(dtm$v),

    Minimum_DF = min_doc_freq,

    Min_Term_Length = MIN_TERM_LENGTH,

    Empty_Processed_Documents = n_invalid,

    Empty_Documents_Removed_After_DF = n_empty,

    Random_Seed = random_seed,

    Burn_In = burn_in,

    Iterations = iterations,

    Thin = thin,

    Top_Terms = top_n,

    Representative_Documents = representative_n,

    Theta_Max_Row_Sum_Error = theta_error,

    Beta_Max_Row_Sum_Error = beta_error,

    Topic_Prevalence_Sum = sum(
        topic_prevalence
    ),

    Elapsed_Seconds = elapsed_seconds,

    stringsAsFactors = FALSE
)


# ============================================================
# Save outputs
# ============================================================

write.csv(
    model_summary,
    summary_file,
    row.names = FALSE
)


write.csv(
    top_terms,
    top_terms_file,
    row.names = FALSE
)


write.csv(
    theta_output,
    theta_file,
    row.names = FALSE
)


write.csv(
    document_topics,
    document_topics_file,
    row.names = FALSE
)


write.csv(
    prevalence,
    prevalence_file,
    row.names = FALSE
)


write.csv(
    representative_documents,
    representative_file,
    row.names = FALSE
)


# ============================================================
# Final diagnostics
# ============================================================

cat("\n")

cat(
    "============================================================\n"
)


cat(
    "FINAL LDA COMPLETE\n"
)


cat(
    "============================================================\n"
)


cat(
    "K:",
    final_k,
    "\n"
)


cat(
    "Documents:",
    nrow(dtm),
    "\n"
)


cat(
    "Vocabulary:",
    ncol(dtm),
    "\n"
)


cat(
    "Tokens:",
    sum(dtm$v),
    "\n"
)


cat(
    "Minimum DF:",
    min_doc_freq,
    "\n"
)


cat(
    "Minimum term length:",
    MIN_TERM_LENGTH,
    "\n"
)


cat(
    "Theta max row-sum error:",
    theta_error,
    "\n"
)


cat(
    "Beta max row-sum error:",
    beta_error,
    "\n"
)


cat(
    "Topic prevalence sum:",
    sum(topic_prevalence),
    "\n"
)


cat(
    "Elapsed seconds:",
    elapsed_seconds,
    "\n"
)


cat(
    "============================================================\n"
)
'''


# ---------------------------------------------------------
# Write R script
# ---------------------------------------------------------

R_FINAL_LDA_SCRIPT.write_text(
    r_final_lda_code,
    encoding="utf-8",
)


print(
    "Created:",
    R_FINAL_LDA_SCRIPT
)


In [ ]:
# ---------------------------------------------------------
# Inspect processed corpus and metadata structure
# ---------------------------------------------------------

for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    processed_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_processed.csv"
    )


    processed_check = pd.read_csv(
        processed_file
    )


    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 70
    )

    print(
        "Processed rows:",
        len(
            processed_check
        )
    )

    print(
        "Processed columns:"
    )

    print(
        processed_check.columns.tolist()
    )

    print()

    print(
        "Corpus rows:",
        len(
            lda_corpora[
                corpus_name
            ]
        )
    )

    print(
        "Corpus columns:"
    )

    print(
        lda_corpora[
            corpus_name
        ].columns.tolist()
    )

In [ ]:
# ---------------------------------------------------------
# Create metadata CSVs for final LDA
# ---------------------------------------------------------

final_metadata_files = {}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    processed_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_processed.csv"
    )


    metadata_file = (
        LDA_DIR
        / f"{corpus_name}_final_metadata.csv"
    )


    processed_check = pd.read_csv(
        processed_file
    )


    metadata = (
        lda_corpora[
            corpus_name
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    # -----------------------------------------------------
    # Strict alignment check
    # -----------------------------------------------------

    if (
        len(
            processed_check
        )
        !=
        len(
            metadata
        )
    ):

        raise ValueError(
            f"Row-count mismatch for {corpus_name}: "
            f"processed={len(processed_check):,}, "
            f"metadata={len(metadata):,}"
        )


    # -----------------------------------------------------
    # Add explicit source-row identifier
    # -----------------------------------------------------

    metadata.insert(
        0,
        "Corpus_Row",
        np.arange(
            1,
            len(
                metadata
            )
            + 1,
        ),
    )


    # -----------------------------------------------------
    # Save
    # -----------------------------------------------------

    metadata.to_csv(
        metadata_file,
        index=False,
    )


    final_metadata_files[
        corpus_name
    ] = metadata_file


    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 60
    )

    print(
        "Processed rows :",
        f"{len(processed_check):,}"
    )

    print(
        "Metadata rows  :",
        f"{len(metadata):,}"
    )

    print(
        "Saved          :",
        metadata_file
    )

In [ ]:
# ---------------------------------------------------------
# Run final full-corpus LDA models
# ---------------------------------------------------------

final_lda_outputs = {}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    # -----------------------------------------------------
    # Inputs
    # -----------------------------------------------------

    processed_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_processed.csv"
    )


    metadata_file = (
        final_metadata_files[
            corpus_name
        ]
    )


    # -----------------------------------------------------
    # Outputs
    # -----------------------------------------------------

    summary_file = (
        LDA_DIR
        / f"{corpus_name}_final_model_summary.csv"
    )


    top_terms_file = (
        LDA_DIR
        / f"{corpus_name}_final_top_terms.csv"
    )


    theta_file = (
        LDA_DIR
        / f"{corpus_name}_final_theta.csv"
    )


    document_topics_file = (
        LDA_DIR
        / f"{corpus_name}_final_document_topics.csv"
    )


    prevalence_file = (
        LDA_DIR
        / f"{corpus_name}_final_topic_prevalence.csv"
    )


    representative_file = (
        LDA_DIR
        / f"{corpus_name}_final_representative_documents.csv"
    )


    log_file = (
        LDA_DIR
        / f"{corpus_name}_final_lda.log"
    )


    # -----------------------------------------------------
    # Configuration
    # -----------------------------------------------------

    final_k = (
        FINAL_K[
            corpus_name
        ]
    )


    min_df = (
        final_min_doc_freq[
            corpus_name
        ]
    )


    # -----------------------------------------------------
    # Report configuration
    # -----------------------------------------------------

    print()
    print(
        "=" * 80
    )

    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 80
    )

    print(
        "Processed corpus :",
        processed_file
    )

    print(
        "Metadata         :",
        metadata_file
    )

    print(
        "Final K          :",
        final_k
    )

    print(
        "Minimum DF       :",
        min_df
    )

    print(
        "Random seed      :",
        FINAL_RANDOM_SEED
    )

    print(
        "Burn-in          :",
        FINAL_BURN_IN
    )

    print(
        "Iterations       :",
        FINAL_ITERATIONS
    )

    print(
        "Thin             :",
        FINAL_THIN
    )

    print(
        "Top terms        :",
        FINAL_TOP_N
    )

    print(
        "Representative   :",
        FINAL_REPRESENTATIVE_N
    )

    print()


    # -----------------------------------------------------
    # Execute R
    # -----------------------------------------------------

    final_run = subprocess.run(
        [
            str(
                RSCRIPT
            ),

            str(
                R_FINAL_LDA_SCRIPT
            ),

            str(
                processed_file
            ),

            str(
                metadata_file
            ),

            str(
                summary_file
            ),

            str(
                top_terms_file
            ),

            str(
                theta_file
            ),

            str(
                document_topics_file
            ),

            str(
                prevalence_file
            ),

            str(
                representative_file
            ),

            str(
                final_k
            ),

            str(
                min_df
            ),

            str(
                FINAL_RANDOM_SEED
            ),

            str(
                FINAL_BURN_IN
            ),

            str(
                FINAL_ITERATIONS
            ),

            str(
                FINAL_THIN
            ),

            str(
                FINAL_TOP_N
            ),

            str(
                FINAL_REPRESENTATIVE_N
            ),
        ],

        capture_output=True,

        text=True,

        check=False,
    )


    # -----------------------------------------------------
    # Save complete R output to log
    # -----------------------------------------------------

    log_text = (
        final_run.stdout
        + "\n"
        + final_run.stderr
    )


    log_file.write_text(
        log_text,
        encoding="utf-8",
    )


    # -----------------------------------------------------
    # Report execution
    # -----------------------------------------------------

    print(
        "Return code:",
        final_run.returncode
    )


    print(
        "Log:",
        log_file
    )


    if final_run.stdout.strip():

        print()
        print(
            "R output:"
        )

        print(
            final_run.stdout
        )


    if final_run.stderr.strip():

        print()
        print(
            "R messages:"
        )

        print(
            final_run.stderr
        )


    # -----------------------------------------------------
    # Stop if failed
    # -----------------------------------------------------

    if (
        final_run.returncode
        != 0
    ):

        raise RuntimeError(
            f"Final LDA failed for "
            f"{corpus_name}.\n"
            f"Inspect:\n{log_file}"
        )


    # -----------------------------------------------------
    # Validate expected output files
    # -----------------------------------------------------

    expected_files = [
        summary_file,
        top_terms_file,
        theta_file,
        document_topics_file,
        prevalence_file,
        representative_file,
    ]


    missing_files = [
        file
        for file
        in expected_files
        if not file.exists()
    ]


    if missing_files:

        raise RuntimeError(
            "Final LDA completed but "
            "expected output files are missing:\n"
            + "\n".join(
                str(file)
                for file
                in missing_files
            )
        )


    # -----------------------------------------------------
    # Store output paths
    # -----------------------------------------------------

    final_lda_outputs[
        corpus_name
    ] = {
        "summary":
            summary_file,

        "top_terms":
            top_terms_file,

        "theta":
            theta_file,

        "document_topics":
            document_topics_file,

        "prevalence":
            prevalence_file,

        "representative_documents":
            representative_file,

        "log":
            log_file,
    }


    print()
    print(
        "Final LDA completed successfully."
    )

In [ ]:
# ---------------------------------------------------------
# Load final LDA outputs
# ---------------------------------------------------------

final_model_summary = {}
final_top_terms = {}
final_theta = {}
final_document_topics = {}
final_topic_prevalence = {}
final_representative_documents = {}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    # -----------------------------------------------------
    # Files
    # -----------------------------------------------------

    summary_file = (
        LDA_DIR
        / f"{corpus_name}_final_model_summary.csv"
    )

    top_terms_file = (
        LDA_DIR
        / f"{corpus_name}_final_top_terms.csv"
    )

    theta_file = (
        LDA_DIR
        / f"{corpus_name}_final_theta.csv"
    )

    document_topics_file = (
        LDA_DIR
        / f"{corpus_name}_final_document_topics.csv"
    )

    prevalence_file = (
        LDA_DIR
        / f"{corpus_name}_final_topic_prevalence.csv"
    )

    representative_file = (
        LDA_DIR
        / f"{corpus_name}_final_representative_documents.csv"
    )


    # -----------------------------------------------------
    # Load
    # -----------------------------------------------------

    final_model_summary[
        corpus_name
    ] = pd.read_csv(
        summary_file
    )

    final_top_terms[
        corpus_name
    ] = pd.read_csv(
        top_terms_file
    )

    final_theta[
        corpus_name
    ] = pd.read_csv(
        theta_file
    )

    final_document_topics[
        corpus_name
    ] = pd.read_csv(
        document_topics_file
    )

    final_topic_prevalence[
        corpus_name
    ] = pd.read_csv(
        prevalence_file
    )

    final_representative_documents[
        corpus_name
    ] = pd.read_csv(
        representative_file
    )


    # -----------------------------------------------------
    # Diagnostics
    # -----------------------------------------------------

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 70
    )

    print(
        "Model summary rows       :",
        len(
            final_model_summary[
                corpus_name
            ]
        )
    )

    print(
        "Top-term rows            :",
        len(
            final_top_terms[
                corpus_name
            ]
        )
    )

    print(
        "Theta shape              :",
        final_theta[
            corpus_name
        ].shape
    )

    print(
        "Document-topic rows      :",
        len(
            final_document_topics[
                corpus_name
            ]
        )
    )

    print(
        "Topic-prevalence rows    :",
        len(
            final_topic_prevalence[
                corpus_name
            ]
        )
    )

    print(
        "Representative-doc rows :",
        len(
            final_representative_documents[
                corpus_name
            ]
        )
    )

    print(
        "Prevalence sum           :",
        final_topic_prevalence[
            corpus_name
        ][
            "Prevalence"
        ].sum()
    )

In [ ]:
# ---------------------------------------------------------
# Build final topic interpretation tables
# ---------------------------------------------------------

final_topic_interpretation = {}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    # -----------------------------------------------------
    # Top terms: one row per topic
    # -----------------------------------------------------

    top_terms_summary = (
        final_top_terms[
            corpus_name
        ]
        .sort_values(
            [
                "Topic",
                "Rank",
            ]
        )
        .groupby(
            "Topic"
        )[
            "Term"
        ]
        .apply(
            lambda values:
                ", ".join(
                    values.astype(str)
                )
        )
        .reset_index(
            name="Top_Terms"
        )
    )


    # -----------------------------------------------------
    # Representative titles: one row per topic
    # -----------------------------------------------------

    representatives = (
        final_representative_documents[
            corpus_name
        ]
        .copy()
    )


    if (
        "Title"
        not in representatives.columns
    ):

        raise ValueError(
            f"Title column missing from "
            f"{corpus_name} representative "
            f"documents."
        )


    representative_titles = (
        representatives
        .sort_values(
            [
                "Topic",
                "Rank",
            ]
        )
        .groupby(
            "Topic"
        )[
            "Title"
        ]
        .apply(
            lambda values:
                " | ".join(
                    values
                    .fillna("")
                    .astype(str)
                )
        )
        .reset_index(
            name="Representative_Titles"
        )
    )


    # -----------------------------------------------------
    # Combine with prevalence
    # -----------------------------------------------------

    interpretation = (
        final_topic_prevalence[
            corpus_name
        ]
        .merge(
            top_terms_summary,
            on="Topic",
            how="left",
        )
        .merge(
            representative_titles,
            on="Topic",
            how="left",
        )
        .sort_values(
            "Prevalence_Rank"
        )
        .reset_index(
            drop=True
        )
    )


    final_topic_interpretation[
        corpus_name
    ] = interpretation


    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 90
    )


    display(
        interpretation[
            [
                "Topic",
                "Prevalence_Rank",
                "Prevalence_Percent",
                "Top_Terms",
                "Representative_Titles",
            ]
        ]
    )

In [ ]:
# ---------------------------------------------------------
# Human-readable final topic interpretation report
# ---------------------------------------------------------

for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    print()
    print()
    print(
        "#" * 100
    )

    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "#" * 100
    )


    interpretation = (
        final_topic_interpretation[
            corpus_name
        ]
        .sort_values(
            "Prevalence_Rank"
        )
    )


    representatives = (
        final_representative_documents[
            corpus_name
        ]
    )


    for topic_row in (
        interpretation
        .itertuples()
    ):

        topic = (
            topic_row.Topic
        )


        print()
        print(
            "=" * 100
        )

        print(
            f"TOPIC {topic} "
            f"| Rank "
            f"{topic_row.Prevalence_Rank} "
            f"| Prevalence "
            f"{topic_row.Prevalence_Percent:.2f}%"
        )

        print(
            "-" * 100
        )

        print(
            "Top terms:"
        )

        print(
            topic_row.Top_Terms
        )

        print()

        print(
            "Representative publications:"
        )


        representative_subset = (
            representatives[
                representatives[
                    "Topic"
                ]
                == topic
            ]
            .sort_values(
                "Rank"
            )
        )


        for row in (
            representative_subset
            .itertuples()
        ):

            print(
                f"  {row.Rank}. "
                f"{row.Title} "
                f"[theta="
                f"{row.Topic_Probability:.3f}]"
            )


        print()

In [ ]:
# ---------------------------------------------------------
# Manual interpretation of final patent/policy LDA topics
# ---------------------------------------------------------

patent_topic_labels = {

    1:  "Distributed voltage optimization and control",

    2:  "Integrated multi-energy systems and decentralized operation",

    3:  "Power-system security and stability assessment",

    4:  "Distributed and nonconvex OPF optimization",

    5:  "State estimation and physics-aware learning",

    6:  "Advanced computing and reliability analysis",

    7:  "Renewable uncertainty and robust dispatch",

    8:  "Electric vehicles, demand, and low-carbon scheduling",

    9:  "Dynamic control and frequency regulation",

    10: "Fault detection and real-time monitoring",

    11: "Deep, graph, and reinforcement learning for grid operation",

    12: "Engineering optimization and device placement",

    13: "Multi-objective optimization and computational intelligence",

    14: "Smart grids, IoT, and digital technologies",

    15: "Data-driven power-system methods",

    16: "Renewable energy management and peer-to-peer markets",

    17: "Artificial intelligence and metaheuristic reviews",

    18: "Prediction and forecasting models",

    19: "General optimization and performance enhancement",

    20: "Metaheuristic and population-based optimization",
}


policy_topic_labels = {

    1:  "Open energy-system modeling and infrastructure analysis",

    2:  "Decarbonization, climate, and energy transition",

    3:  "Renewable generation, storage, and electrified transport",

    4:  "Optimal power flow and mathematical optimization",

    5:  "Uncertainty-aware planning and dispatch",

    6:  "Computational and metaheuristic optimization",

    7:  "Digitalization, cybersecurity, and emerging technologies",

    8:  "Energy-system reviews and research directions",

    9:  "Power-system optimization and adaptive methods",

    10: "Cybersecurity and anomaly detection",

    11: "Forecasting and predictive analytics",

    12: "State, parameter, and topology estimation",

    13: "Grid reliability, transmission, and capacity planning",

    14: "Smart grids, electric vehicles, and IoT",

    15: "Distributed voltage control and DER coordination",

    16: "Grid resilience and critical infrastructure",

    17: "Renewable energy management and storage integration",

    18: "Microgrid control, protection, and resilience",

    19: "Electricity markets and peer-to-peer energy trading",

    20: "Data analytics, privacy, and smart metering",
}

In [ ]:
# ---------------------------------------------------------
# Attach manual topic labels
# ---------------------------------------------------------

topic_label_maps = {
    "patent_cited":
        patent_topic_labels,

    "policy_cited":
        policy_topic_labels,
}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    final_topic_interpretation[
        corpus_name
    ][
        "Topic_Label"
    ] = (
        final_topic_interpretation[
            corpus_name
        ][
            "Topic"
        ]
        .map(
            topic_label_maps[
                corpus_name
            ]
        )
    )


    # Validation
    missing_labels = (
        final_topic_interpretation[
            corpus_name
        ][
            "Topic_Label"
        ]
        .isna()
        .sum()
    )


    if missing_labels != 0:

        raise ValueError(
            f"{missing_labels} topic labels "
            f"missing for {corpus_name}."
        )


    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 80
    )


    display(
        final_topic_interpretation[
            corpus_name
        ][
            [
                "Topic",
                "Prevalence_Rank",
                "Prevalence_Percent",
                "Topic_Label",
                "Top_Terms",
            ]
        ]
        .sort_values(
            "Prevalence_Rank"
        )
    )

In [ ]:
# ---------------------------------------------------------
# Save final interpreted topic tables
# ---------------------------------------------------------

for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    output_file = (
        LDA_DIR
        / f"{corpus_name}_final_topic_interpretation.csv"
    )


    final_topic_interpretation[
        corpus_name
    ].to_csv(
        output_file,
        index=False,
    )


    print(
        "Saved:",
        output_file
    )

In [ ]:
# ---------------------------------------------------------
# Common cross-corpus macro-themes
# ---------------------------------------------------------

MACRO_THEMES = [

    "OPF and Mathematical Optimization",

    "Metaheuristics and Computational Optimization",

    "Machine Learning, Forecasting, and Predictive Analytics",

    "State Estimation, Monitoring, and Cybersecurity",

    "Distributed Control and Grid Operation",

    "Renewable Energy, Storage, and Uncertainty",

    "Energy Management, Markets, and Demand Flexibility",

    "Smart Grids, IoT, and Digitalization",

    "Reliability, Resilience, Security, and Stability",

    "Decarbonization, Electrification, and Integrated Energy Systems",
]

In [ ]:
# ---------------------------------------------------------
# Patent topics -> common macro-themes
# ---------------------------------------------------------

patent_macro_theme_map = {

    1: "Distributed Control and Grid Operation",

    2: "Decarbonization, Electrification, and Integrated Energy Systems",

    3: "Reliability, Resilience, Security, and Stability",

    4: "OPF and Mathematical Optimization",

    5: "State Estimation, Monitoring, and Cybersecurity",

    6: "Reliability, Resilience, Security, and Stability",

    7: "Renewable Energy, Storage, and Uncertainty",

    8: "Decarbonization, Electrification, and Integrated Energy Systems",

    9: "Distributed Control and Grid Operation",

    10: "State Estimation, Monitoring, and Cybersecurity",

    11: "Machine Learning, Forecasting, and Predictive Analytics",

    12: "OPF and Mathematical Optimization",

    13: "Metaheuristics and Computational Optimization",

    14: "Smart Grids, IoT, and Digitalization",

    15: "Machine Learning, Forecasting, and Predictive Analytics",

    16: "Energy Management, Markets, and Demand Flexibility",

    17: "Metaheuristics and Computational Optimization",

    18: "Machine Learning, Forecasting, and Predictive Analytics",

    19: "Metaheuristics and Computational Optimization",

    20: "Metaheuristics and Computational Optimization",
}

In [ ]:
# ---------------------------------------------------------
# Policy topics -> common macro-themes
# ---------------------------------------------------------

policy_macro_theme_map = {

    1: "Decarbonization, Electrification, and Integrated Energy Systems",

    2: "Decarbonization, Electrification, and Integrated Energy Systems",

    3: "Renewable Energy, Storage, and Uncertainty",

    4: "OPF and Mathematical Optimization",

    5: "Renewable Energy, Storage, and Uncertainty",

    6: "Metaheuristics and Computational Optimization",

    7: "Smart Grids, IoT, and Digitalization",

    8: "Decarbonization, Electrification, and Integrated Energy Systems",

    9: "OPF and Mathematical Optimization",

    10: "State Estimation, Monitoring, and Cybersecurity",

    11: "Machine Learning, Forecasting, and Predictive Analytics",

    12: "State Estimation, Monitoring, and Cybersecurity",

    13: "Reliability, Resilience, Security, and Stability",

    14: "Smart Grids, IoT, and Digitalization",

    15: "Distributed Control and Grid Operation",

    16: "Reliability, Resilience, Security, and Stability",

    17: "Energy Management, Markets, and Demand Flexibility",

    18: "Reliability, Resilience, Security, and Stability",

    19: "Energy Management, Markets, and Demand Flexibility",

    20: "State Estimation, Monitoring, and Cybersecurity",
}

In [ ]:
# ---------------------------------------------------------
# Validate cross-corpus mapping
# ---------------------------------------------------------

for corpus_name, mapping in [

    (
        "patent_cited",
        patent_macro_theme_map,
    ),

    (
        "policy_cited",
        policy_macro_theme_map,
    ),
]:

    expected_topics = set(
        range(
            1,
            FINAL_K[
                corpus_name
            ] + 1
        )
    )

    mapped_topics = set(
        mapping.keys()
    )


    missing_topics = (
        expected_topics
        - mapped_topics
    )

    extra_topics = (
        mapped_topics
        - expected_topics
    )


    invalid_themes = set(
        mapping.values()
    ) - set(
        MACRO_THEMES
    )


    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 60
    )

    print(
        "Topics expected :",
        len(
            expected_topics
        )
    )

    print(
        "Topics mapped   :",
        len(
            mapped_topics
        )
    )

    print(
        "Missing topics  :",
        sorted(
            missing_topics
        )
    )

    print(
        "Extra topics    :",
        sorted(
            extra_topics
        )
    )

    print(
        "Invalid themes  :",
        sorted(
            invalid_themes
        )
    )


    if (
        missing_topics
        or
        extra_topics
        or
        invalid_themes
    ):

        raise ValueError(
            f"Invalid macro-theme mapping "
            f"for {corpus_name}."
        )

In [ ]:
# ---------------------------------------------------------
# Attach macro-themes
# ---------------------------------------------------------

macro_theme_maps = {

    "patent_cited":
        patent_macro_theme_map,

    "policy_cited":
        policy_macro_theme_map,
}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    final_topic_interpretation[
        corpus_name
    ][
        "Macro_Theme"
    ] = (
        final_topic_interpretation[
            corpus_name
        ][
            "Topic"
        ]
        .map(
            macro_theme_maps[
                corpus_name
            ]
        )
    )


    display(
        final_topic_interpretation[
            corpus_name
        ][
            [
                "Topic",
                "Topic_Label",
                "Macro_Theme",
                "Prevalence_Percent",
            ]
        ]
        .sort_values(
            "Macro_Theme"
        )
    )

In [ ]:
# ---------------------------------------------------------
# Aggregate macro-theme prevalence
# ---------------------------------------------------------

macro_theme_prevalence = {}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    summary = (
        final_topic_interpretation[
            corpus_name
        ]
        .groupby(
            "Macro_Theme",
            as_index=False,
        )[
            "Prevalence_Percent"
        ]
        .sum()
    )


    # Ensure all ten themes are represented
    summary = (
        pd.DataFrame(
            {
                "Macro_Theme":
                    MACRO_THEMES
            }
        )
        .merge(
            summary,
            on="Macro_Theme",
            how="left",
        )
        .fillna(
            {
                "Prevalence_Percent":
                    0.0
            }
        )
    )


    macro_theme_prevalence[
        corpus_name
    ] = summary


    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 80
    )


    display(
        summary
        .sort_values(
            "Prevalence_Percent",
            ascending=False,
        )
    )


    print(
        "Total prevalence:",
        f"{summary['Prevalence_Percent'].sum():.6f}%"
    )

In [ ]:
# ---------------------------------------------------------
# Patent-policy macro-theme comparison
# ---------------------------------------------------------

patent_macro = (
    macro_theme_prevalence[
        "patent_cited"
    ]
    .rename(
        columns={
            "Prevalence_Percent":
                "Patent_Prevalence"
        }
    )
)


policy_macro = (
    macro_theme_prevalence[
        "policy_cited"
    ]
    .rename(
        columns={
            "Prevalence_Percent":
                "Policy_Prevalence"
        }
    )
)


macro_theme_comparison = (
    patent_macro
    .merge(
        policy_macro,
        on="Macro_Theme",
        how="outer",
    )
    .fillna(0)
)


macro_theme_comparison[
    "Difference_pp"
] = (
    macro_theme_comparison[
        "Patent_Prevalence"
    ]
    -
    macro_theme_comparison[
        "Policy_Prevalence"
    ]
)


macro_theme_comparison[
    "Absolute_Difference_pp"
] = (
    macro_theme_comparison[
        "Difference_pp"
    ]
    .abs()
)


macro_theme_comparison[
    "Patent_to_Policy_Ratio"
] = np.where(
    macro_theme_comparison[
        "Policy_Prevalence"
    ] > 0,

    macro_theme_comparison[
        "Patent_Prevalence"
    ]
    /
    macro_theme_comparison[
        "Policy_Prevalence"
    ],

    np.nan,
)


macro_theme_comparison = (
    macro_theme_comparison
    .sort_values(
        "Absolute_Difference_pp",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)

# ---------------------------------------------------------
# Save patent-policy macro-theme comparison
# ---------------------------------------------------------

PATENT_POLICY_MACRO_FILE = (
    LDA_DIR
    / "patent_policy_macro_theme_table.csv"
)


macro_theme_comparison.to_csv(
    PATENT_POLICY_MACRO_FILE,
    index=False,
)


print(
    "Saved:",
    PATENT_POLICY_MACRO_FILE
)


display(
    macro_theme_comparison.round(2)
)

In [ ]:
# ---------------------------------------------------------
# Diverging macro-theme prevalence comparison
# Two colors: patent-oriented vs policy-oriented
# ---------------------------------------------------------

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch


# ---------------------------------------------------------
# Prepare data
# ---------------------------------------------------------

plot_data = (
    macro_theme_comparison
    .copy()
    .sort_values(
        "Difference_pp"
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# Publication-friendly display labels
# ---------------------------------------------------------

display_labels = {

    "OPF and Mathematical Optimization":
        "OPF and mathematical optimization",

    "Metaheuristics and Computational Optimization":
        "Metaheuristics and computational optimization",

    "Machine Learning, Forecasting, and Predictive Analytics":
        "ML, forecasting, and predictive analytics",

    "State Estimation, Monitoring, and Cybersecurity":
        "State estimation, monitoring, and cybersecurity",

    "Distributed Control and Grid Operation":
        "Distributed control and grid operation",

    "Renewable Energy, Storage, and Uncertainty":
        "Renewables, storage, and uncertainty",

    "Energy Management, Markets, and Demand Flexibility":
        "Energy management, markets, and demand flexibility",

    "Smart Grids, IoT, and Digitalization":
        "Smart grids, IoT, and digitalization",

    "Reliability, Resilience, Security, and Stability":
        "Reliability, resilience, security, and stability",

    "Decarbonization, Electrification, and Integrated Energy Systems":
        "Decarbonization, electrification, and integrated energy systems",
}


plot_data[
    "Display_Label"
] = (
    plot_data[
        "Macro_Theme"
    ]
    .map(
        display_labels
    )
)


# ---------------------------------------------------------
# Colors
# ---------------------------------------------------------

PATENT_COLOR = "#287DB5"
POLICY_COLOR = "#E07A3F"


bar_colors = np.where(
    plot_data[
        "Difference_pp"
    ] >= 0,
    PATENT_COLOR,
    POLICY_COLOR,
)


# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(
        11.5,
        6.7,
    )
)


y = np.arange(
    len(
        plot_data
    )
)


bars = ax.barh(
    y,
    plot_data[
        "Difference_pp"
    ],
    color=bar_colors,
    height=0.78,
    zorder=3,
)


# ---------------------------------------------------------
# Zero reference line
# ---------------------------------------------------------

ax.axvline(
    0,
    color="black",
    linewidth=1.0,
    zorder=4,
)


# ---------------------------------------------------------
# Value labels
# ---------------------------------------------------------

for bar, value in zip(
    bars,
    plot_data[
        "Difference_pp"
    ],
):

    y_position = (
        bar.get_y()
        +
        bar.get_height() / 2
    )


    if value > 0:

        ax.text(
            value + 0.25,
            y_position,
            f"+{value:.2f}",
            va="center",
            ha="left",
            fontsize=9.5,
        )

    else:

        ax.text(
            value - 0.25,
            y_position,
            f"{value:.2f}",
            va="center",
            ha="right",
            fontsize=9.5,
        )


# ---------------------------------------------------------
# Y axis
# ---------------------------------------------------------

ax.set_yticks(
    y
)


ax.set_yticklabels(
    plot_data[
        "Display_Label"
    ],
    fontsize=9.5,
)


ax.set_ylabel(
    ""
)


# ---------------------------------------------------------
# X axis
# ---------------------------------------------------------

ax.set_xlabel(
    "Prevalence difference (percentage points)\n"
    "Patent-cited minus policy-cited",
    fontsize=10.5,
)


# Give labels sufficient room
x_min = (
    plot_data[
        "Difference_pp"
    ].min()
    - 1.5
)

x_max = (
    plot_data[
        "Difference_pp"
    ].max()
    + 2.0
)


ax.set_xlim(
    x_min,
    x_max,
)


# ---------------------------------------------------------
# Directional annotations
# Keep them inside the axes, below the title
# ---------------------------------------------------------

# ax.text(
#     0.01,
#     0.985,
#     "MORE PREVALENT IN\nPOLICY-CITED RESEARCH",
#     transform=ax.transAxes,
#     ha="left",
#     va="top",
#     fontsize=9,
#     fontweight="bold",
#     color=POLICY_COLOR,
# )


# ax.text(
#     0.99,
#     0.985,
#     "MORE PREVALENT IN\nPATENT-CITED RESEARCH",
#     transform=ax.transAxes,
#     ha="right",
#     va="top",
#     fontsize=9,
#     fontweight="bold",
#     color=PATENT_COLOR,
# )


# ---------------------------------------------------------
# Grid
# ---------------------------------------------------------

ax.grid(
    axis="x",
    linestyle="-",
    linewidth=0.6,
    alpha=0.18,
    zorder=0,
)


# ---------------------------------------------------------
# Spines
# ---------------------------------------------------------

ax.spines[
    "top"
].set_visible(
    False
)


ax.spines[
    "right"
].set_visible(
    False
)


# ---------------------------------------------------------
# Legend
# ---------------------------------------------------------

legend_handles = [

    Patch(
        facecolor=PATENT_COLOR,
        label="More prevalent in patent-cited research",
    ),

    Patch(
        facecolor=POLICY_COLOR,
        label="More prevalent in policy-cited research",
    ),
]


ax.legend(
    handles=legend_handles,
    loc="lower right",
    frameon=False,
    fontsize=8.5,
)


# ---------------------------------------------------------
# UPPERCASE title
# ---------------------------------------------------------

ax.set_title(
    "COMPARATIVE PREVALENCE OF PATENT- AND POLICY-CITED RESEARCH THEMES",
    fontsize=12,
    fontweight="bold",
    loc="left",
    pad=18,
)


# ---------------------------------------------------------
# Layout
# ---------------------------------------------------------

fig.subplots_adjust(
    left=0.39,
    right=0.97,
    top=0.90,
    bottom=0.15,
)


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

macro_difference_figure = (
    FIGURE_DIR
    / "patent_policy_macro_theme_difference.pdf"
)


fig.savefig(
    macro_difference_figure,
    format="pdf",
    bbox_inches="tight",
)


macro_difference_png = (
    FIGURE_DIR
    / "patent_policy_macro_theme_difference.png"
)


fig.savefig(
    macro_difference_png,
    dpi=600,
    bbox_inches="tight",
)


print(
    "Saved PDF:",
    macro_difference_figure
)


print(
    "Saved PNG:",
    macro_difference_png
)


plt.show()

In [ ]:
# ---------------------------------------------------------
# Publication-quality dumbbell plot:
# Patent- vs policy-cited macro-theme prevalence
# ---------------------------------------------------------

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ---------------------------------------------------------
# Prepare data
# ---------------------------------------------------------

plot_data = (
    macro_theme_comparison
    .copy()
)


# Shorter labels for publication figure
display_labels = {

    "OPF and Mathematical Optimization":
        "OPF and mathematical optimization",

    "Metaheuristics and Computational Optimization":
        "Metaheuristics and computational optimization",

    "Machine Learning, Forecasting, and Predictive Analytics":
        "ML, forecasting, and predictive analytics",

    "State Estimation, Monitoring, and Cybersecurity":
        "State estimation, monitoring, and cybersecurity",

    "Distributed Control and Grid Operation":
        "Distributed control and grid operation",

    "Renewable Energy, Storage, and Uncertainty":
        "Renewables, storage, and uncertainty",

    "Energy Management, Markets, and Demand Flexibility":
        "Energy management, markets, and demand flexibility",

    "Smart Grids, IoT, and Digitalization":
        "Smart grids, IoT, and digitalization",

    "Reliability, Resilience, Security, and Stability":
        "Reliability, resilience, security, and stability",

    "Decarbonization, Electrification, and Integrated Energy Systems":
        "Decarbonization, electrification,\nand integrated energy systems",
}


plot_data[
    "Display_Label"
] = (
    plot_data[
        "Macro_Theme"
    ]
    .map(
        display_labels
    )
)


# ---------------------------------------------------------
# Sort by average prevalence
# This puts the most prominent themes toward the top.
# ---------------------------------------------------------

plot_data[
    "Mean_Prevalence"
] = (
    plot_data[
        [
            "Patent_Prevalence",
            "Policy_Prevalence",
        ]
    ]
    .mean(
        axis=1
    )
)


plot_data = (
    plot_data
    .sort_values(
        "Mean_Prevalence",
        ascending=True,
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# Plot geometry
# ---------------------------------------------------------

y = np.arange(
    len(
        plot_data
    )
)


patent_values = (
    plot_data[
        "Patent_Prevalence"
    ]
    .to_numpy()
)


policy_values = (
    plot_data[
        "Policy_Prevalence"
    ]
    .to_numpy()
)


fig, ax = plt.subplots(
    figsize=(
        11.5,
        7.0,
    )
)


# ---------------------------------------------------------
# Dumbbell connectors
# ---------------------------------------------------------

for i in range(
    len(
        plot_data
    )
):

    ax.plot(
        [
            policy_values[i],
            patent_values[i],
        ],
        [
            y[i],
            y[i],
        ],
        linewidth=2.0,
        alpha=0.55,
        zorder=1,
    )


# ---------------------------------------------------------
# End-point markers
# ---------------------------------------------------------

ax.scatter(
    policy_values,
    y,
    s=75,
    marker="o",
    label="Policy-cited research",
    zorder=3,
)


ax.scatter(
    patent_values,
    y,
    s=75,
    marker="s",
    label="Patent-cited research",
    zorder=3,
)


# ---------------------------------------------------------
# Numeric labels
# ---------------------------------------------------------

label_offset = 0.25


for i in range(
    len(
        plot_data
    )
):

    patent_value = (
        patent_values[i]
    )

    policy_value = (
        policy_values[i]
    )


    # -----------------------------------------------------
    # Avoid overlap when values are very close
    # -----------------------------------------------------

    if abs(
        patent_value
        - policy_value
    ) < 1.0:

        # Policy above
        ax.annotate(
            f"{policy_value:.1f}%",
            (
                policy_value,
                y[i],
            ),
            xytext=(
                0,
                9,
            ),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8.5,
        )


        # Patent below
        ax.annotate(
            f"{patent_value:.1f}%",
            (
                patent_value,
                y[i],
            ),
            xytext=(
                0,
                -10,
            ),
            textcoords="offset points",
            ha="center",
            va="top",
            fontsize=8.5,
        )


    else:

        # -------------------------------------------------
        # Put labels outward from the connector
        # -------------------------------------------------

        if patent_value > policy_value:

            ax.text(
                patent_value
                + label_offset,
                y[i],
                f"{patent_value:.1f}%",
                va="center",
                ha="left",
                fontsize=8.5,
            )


            ax.text(
                policy_value
                - label_offset,
                y[i],
                f"{policy_value:.1f}%",
                va="center",
                ha="right",
                fontsize=8.5,
            )


        else:

            ax.text(
                policy_value
                + label_offset,
                y[i],
                f"{policy_value:.1f}%",
                va="center",
                ha="left",
                fontsize=8.5,
            )


            ax.text(
                patent_value
                - label_offset,
                y[i],
                f"{patent_value:.1f}%",
                va="center",
                ha="right",
                fontsize=8.5,
            )


# ---------------------------------------------------------
# Y-axis
# ---------------------------------------------------------

ax.set_yticks(
    y
)


ax.set_yticklabels(
    plot_data[
        "Display_Label"
    ],
    fontsize=9.5,
)


ax.set_ylabel(
    ""
)


# ---------------------------------------------------------
# X-axis
# ---------------------------------------------------------

maximum_value = max(
    patent_values.max(),
    policy_values.max(),
)


ax.set_xlim(
    0,
    maximum_value
    + 2.2,
)


ax.set_xlabel(
    "Theme prevalence within corpus (%)",
    fontsize=10.5,
)


# ---------------------------------------------------------
# Grid
# ---------------------------------------------------------

ax.grid(
    axis="x",
    linestyle="--",
    linewidth=0.7,
    alpha=0.25,
)


ax.grid(
    axis="y",
    visible=False,
)


# ---------------------------------------------------------
# Spines
# ---------------------------------------------------------

ax.spines[
    "top"
].set_visible(
    False
)


ax.spines[
    "right"
].set_visible(
    False
)


ax.spines[
    "left"
].set_visible(
    False
)


# ---------------------------------------------------------
# Legend
# ---------------------------------------------------------

ax.legend(
    loc="lower right",
    frameon=False,
    fontsize=9.5,
)


# ---------------------------------------------------------
# Title
# ---------------------------------------------------------

ax.set_title(
    "Comparative Prevalence of Themes in Patent- and Policy-Cited Research",
    fontsize=12,
    fontweight="bold",
    loc="left",
    pad=14,
)


# ---------------------------------------------------------
# Layout
# ---------------------------------------------------------

fig.subplots_adjust(
    left=0.36,
    right=0.97,
    top=0.91,
    bottom=0.11,
)


# ---------------------------------------------------------
# Save PDF
# ---------------------------------------------------------

dumbbell_figure = (
    FIGURE_DIR
    / "patent_policy_macro_theme_dumbbell.pdf"
)


fig.savefig(
    dumbbell_figure,
    format="pdf",
    bbox_inches="tight",
)


# ---------------------------------------------------------
# Also save high-resolution PNG for inspection
# ---------------------------------------------------------

dumbbell_png = (
    FIGURE_DIR
    / "patent_policy_macro_theme_dumbbell.png"
)


fig.savefig(
    dumbbell_png,
    dpi=600,
    bbox_inches="tight",
)


print(
    "Saved PDF:",
    dumbbell_figure
)


print(
    "Saved PNG:",
    dumbbell_png
)


plt.show()